# Cafe Data Analytics Data Cleaning and Transformation

In [106]:
import pandas as pd
import warnings
import json
import numpy as np
from datetime import datetime, date
from tabulate import tabulate

In [2]:
warnings.filterwarnings('ignore')

In [3]:
pd.set_option('max_colwidth', 2000)

## Load Raw Data from BigQuery

In [4]:
# Load data from BigQuery into local notebook
df_raw = pd.read_gbq(
    """
        SELECT *
        FROM `jr-data-training.cafe.cafe-sales`
    """,
    project_id='jr-data-training',
    location='australia-southeast1',
)

In [5]:
# Check the first 5 rows of the raw data
df_raw.head()

,order_id,customer_id,ip_addr,date_created,date_paid,total,status,items
0,11787,517,81741da2cc0e5207bfefbb4ec257b2393d5e13a0,2021-04-14 05:56:06,2021-04-14 05:56:37,1280,2,"{""cart_size"":2,""cart_surcharge"":0,""cart_total_price"":1280,""cart_gst"":116.3636363636363597606759867630898952484130859375,""cart_surcharge_display"":""$0.00"",""cart_total_price_display"":""$12.80"",""cart_gst_display"":""$1.16"",""cart"":[{""name"":""Latte"",""variant_name"":""Large"",""variant_desc"":"""",""variant_image"":""images\/coffeecup-latte.png"",""options"":[{""name"":""size"",""value"":""LRG"",""price"":430},{""name"":""Milk"",""value"":""Soy"",""price"":50},{""name"":""Strength"",""value"":""Full"",""price"":0},{""name"":""Decaf"",""value"":""Normal"",""price"":0},{""name"":""Temp"",""value"":""Normal"",""price"":0},{""name"":""Honey"",""value"":""None"",""price"":0},{""name"":""Syrup"",""value"":""None"",""price"":0},{""name"":""White Sugar"",""value"":""0"",""price"":0},{""name"":""Raw Sugar"",""value"":""0"",""price"":0},{""name"":""Equal Sugar"",""value"":""0"",""price"":0},{""name"":""Extra shot"",""value"":""0"",""price"":0}],""price"":480,""category"":""Hot Drinks""},{""name"":""Toastie"",""variant_name"":""Bacon, Egg and Cheese"",""variant_desc"":"""",""variant_image"":""images\/toastie-baconegg.png"",""options"":[{""name"":""size"",""value"":""BACON-EGG-CHEESE"",""price"":800},{""name"":""Bread"",""value"":""Multigrain"",""price"":0}],""price"":800,""category"":""Kitchen""}],""order_phone"":""**********"",""order_name"":""********"",""order_time"":""1618344900"",""order_time_verbose"":""6:15am""}"
1,40606,519,ccf5e36bddfbce1a7de2c04874190a057b30588d,2024-03-05 10:16:40,2024-03-05 10:17:23,1280,2,"{""cart_size"":2,""cart_surcharge"":0,""cart_total_price"":1280,""cart_gst"":116.36363636363636,""cart_surcharge_display"":""$0.00"",""cart_total_price_display"":""$12.80"",""cart_gst_display"":""$1.16"",""cart"":[{""name"":""Flat White"",""variant_name"":""Extra Large"",""variant_desc"":"""",""variant_image"":""images\/coffee-flatwhite.png"",""options"":[{""name"":""size"",""value"":""XLG"",""price"":540},{""name"":""Milk"",""value"":""Lactose Free"",""price"":50},{""name"":""Strength"",""value"":""Full"",""price"":0},{""name"":""Decaf"",""value"":""Normal"",""price"":0},{""name"":""Temp"",""value"":""Normal"",""price"":0},{""name"":""Honey"",""value"":""None"",""price"":0},{""name"":""Syrup"",""value"":""None"",""price"":0},{""name"":""White Sugar"",""value"":""0"",""price"":0},{""name"":""Raw Sugar"",""value"":""0"",""price"":0},{""name"":""Equal Sugar"",""value"":""0"",""price"":0},{""name"":""Extra shot"",""value"":""1"",""price"":50}],""unitprice"":640,""price"":640,""category"":""Hot Drinks"",""quantity"":1},{""name"":""Latte"",""variant_name"":""Extra Large"",""variant_desc"":"""",""variant_image"":""images\/coffeecup-latte.png"",""options"":[{""name"":""size"",""value"":""XLG"",""price"":540},{""name"":""Milk"",""value"":""Lactose Free"",""price"":50},{""name"":""Strength"",""value"":""Full"",""price"":0},{""name"":""Decaf"",""value"":""Normal"",""price"":0},{""name"":""Temp"",""value"":""Normal"",""price"":0},{""name"":""Honey"",""value"":""None"",""price"":0},{""name"":""Syrup"",""value"":""None"",""price"":0},{""name"":""White Sugar"",""value"":""0"",""price"":0},{""name"":""Raw Sugar"",""value"":""0"",""price"":0},{""name"":""Equal Sugar"",""value"":""0"",""price"":0},{""name"":""Extra shot"",""value"":""1"",""price"":50}],""unitprice"":640,""price"":640,""category"":""Hot Drinks"",""quantity"":1}],""order_phone"":""**********"",""order_name"":""********"",""order_time"":""ASAP"",""order_time_verbose"":""ASAP""}"
2,40488,519,ccf5e36bddfbce1a7de2c04874190a057b30588d,2024-03-01 11:40:20,2024-03-01 11:40:58,1280,2,"{""cart_size"":2,""cart_surcharge"":0,""cart_total_price"":1280,""cart_gst"":116.36363636363636,""cart_surcharge_display"":""$0.00"",""cart_total_price_display"":""$12.80"",""cart_gst_display"":""$1.16"",""cart"":[{"

In [6]:
# Check the schema of raw data
print(df_raw.shape)
print(df_raw.info())

(37372, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37372 entries, 0 to 37371
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      37372 non-null  Int64         
 1   customer_id   37372 non-null  Int64         
 2   ip_addr       37372 non-null  object        
 3   date_created  37372 non-null  datetime64[ns]
 4   date_paid     36019 non-null  datetime64[ns]
 5   total         37372 non-null  Int64         
 6   status        37372 non-null  Int64         
 7   items         37372 non-null  object        
dtypes: Int64(4), datetime64[ns](2), object(2)
memory usage: 2.4+ MB
None


The `items` column contains JSON strings of order details. We need to extract fields from this column.

## Data Cleaning and Transformation

In [245]:
df = df_raw.copy()

### Remove Unpaid Orders

In [246]:
df = df[
    ~pd.isna(df['date_paid'])
]

In [247]:
print(df.shape)
print(df.info())

(36019, 8)
<class 'pandas.core.frame.DataFrame'>
Int64Index: 36019 entries, 0 to 37371
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      36019 non-null  Int64         
 1   customer_id   36019 non-null  Int64         
 2   ip_addr       36019 non-null  object        
 3   date_created  36019 non-null  datetime64[ns]
 4   date_paid     36019 non-null  datetime64[ns]
 5   total         36019 non-null  Int64         
 6   status        36019 non-null  Int64         
 7   items         36019 non-null  object        
dtypes: Int64(4), datetime64[ns](2), object(2)
memory usage: 2.6+ MB
None


### Extract field values

In [248]:
# Try converting the JSON string in 'items' column to dict type
try:
    df['items'] = df['items'].apply(lambda x: json.loads(x))
except Exception as e:
    print(e)

In [249]:
# View the first items value
df['items'][0]

{'cart_size': 2,
 'cart_surcharge': 0,
 'cart_total_price': 1280,
 'cart_gst': 116.36363636363636,
 'cart_surcharge_display': '$0.00',
 'cart_total_price_display': '$12.80',
 'cart_gst_display': '$1.16',
 'cart': [{'name': 'Latte',
   'variant_name': 'Large',
   'variant_desc': '',
   'variant_image': 'images/coffeecup-latte.png',
   'options': [{'name': 'size', 'value': 'LRG', 'price': 430},
    {'name': 'Milk', 'value': 'Soy', 'price': 50},
    {'name': 'Strength', 'value': 'Full', 'price': 0},
    {'name': 'Decaf', 'value': 'Normal', 'price': 0},
    {'name': 'Temp', 'value': 'Normal', 'price': 0},
    {'name': 'Honey', 'value': 'None', 'price': 0},
    {'name': 'Syrup', 'value': 'None', 'price': 0},
    {'name': 'White Sugar', 'value': '0', 'price': 0},
    {'name': 'Raw Sugar', 'value': '0', 'price': 0},
    {'name': 'Equal Sugar', 'value': '0', 'price': 0},
    {'name': 'Extra shot', 'value': '0', 'price': 0}],
   'price': 480,
   'category': 'Hot Drinks'},
  {'name': 'Toastie',


In [250]:
# Extract preliminary fields from the 'items' column
for field_name in df['items'][0].keys():
    df[field_name] = df['items'].str[field_name]

In [251]:
df.columns

Index(['order_id', 'customer_id', 'ip_addr', 'date_created', 'date_paid',
       'total', 'status', 'items', 'cart_size', 'cart_surcharge',
       'cart_total_price', 'cart_gst', 'cart_surcharge_display',
       'cart_total_price_display', 'cart_gst_display', 'cart', 'order_phone',
       'order_name', 'order_time', 'order_time_verbose'],
      dtype='object')

In [252]:
# The 'cart' column contains lists of ordered items
df.loc[0, 'cart']

[{'name': 'Latte',
  'variant_name': 'Large',
  'variant_desc': '',
  'variant_image': 'images/coffeecup-latte.png',
  'options': [{'name': 'size', 'value': 'LRG', 'price': 430},
   {'name': 'Milk', 'value': 'Soy', 'price': 50},
   {'name': 'Strength', 'value': 'Full', 'price': 0},
   {'name': 'Decaf', 'value': 'Normal', 'price': 0},
   {'name': 'Temp', 'value': 'Normal', 'price': 0},
   {'name': 'Honey', 'value': 'None', 'price': 0},
   {'name': 'Syrup', 'value': 'None', 'price': 0},
   {'name': 'White Sugar', 'value': '0', 'price': 0},
   {'name': 'Raw Sugar', 'value': '0', 'price': 0},
   {'name': 'Equal Sugar', 'value': '0', 'price': 0},
   {'name': 'Extra shot', 'value': '0', 'price': 0}],
  'price': 480,
  'category': 'Hot Drinks'},
 {'name': 'Toastie',
  'variant_name': 'Bacon, Egg and Cheese',
  'variant_desc': '',
  'variant_image': 'images/toastie-baconegg.png',
  'options': [{'name': 'size', 'value': 'BACON-EGG-CHEESE', 'price': 800},
   {'name': 'Bread', 'value': 'Multigrai

We need to break down the lists in the `cart` column so that each purchased item in an order is placed in its own row.

In [253]:
df['cart'].apply(lambda x: type(x)).value_counts()

<class 'list'>    35249
<class 'dict'>      770
Name: cart, dtype: int64

The `cart` column contains not only list values but also dict values.

In [254]:
# Sample a 'cart' value that is in dict type
df['cart'].apply(
    lambda x: x if type(x) == dict else None
).value_counts().index[0]

{'1': {'name': 'Cappuccino',
  'variant_name': 'Large',
  'variant_desc': '',
  'variant_image': 'images/coffeemug-cappuccino.png',
  'options': [{'name': 'size', 'value': 'LRG', 'price': 480},
   {'name': 'Milk', 'value': 'Full Cream', 'price': 0},
   {'name': 'Strength', 'value': 'Full', 'price': 0},
   {'name': 'Decaf', 'value': 'Normal', 'price': 0},
   {'name': 'Temp', 'value': 'Normal', 'price': 0},
   {'name': 'Honey', 'value': 'None', 'price': 0},
   {'name': 'Syrup', 'value': 'None', 'price': 0},
   {'name': 'White Sugar', 'value': '0', 'price': 0},
   {'name': 'Raw Sugar', 'value': '0', 'price': 0},
   {'name': 'Equal Sugar', 'value': '0', 'price': 0},
   {'name': 'Extra shot', 'value': '0', 'price': 0}],
  'unitprice': 480,
  'price': 480,
  'category': 'Hot Drinks',
  'quantity': 1}}

In [255]:
def convert_to_list(x):
    if type(x) == dict:
        return list(x.values())
    else:
        return x
    
df['cart'] = df['cart'].apply(convert_to_list)

In [256]:
df['cart'].apply(lambda x: type(x)).value_counts()

<class 'list'>    36019
Name: cart, dtype: int64

In [257]:
# Separate elements in the `cart` array into multiple rows
df = df.explode('cart', ignore_index=True)
df[['order_id', 'cart']].head()

,order_id,cart
0,11787,"{'name': 'Latte', 'variant_name': 'Large', 'variant_desc': '', 'variant_image': 'images/coffeecup-latte.png', 'options': [{'name': 'size', 'value': 'LRG', 'price': 430}, {'name': 'Milk', 'value': 'Soy', 'price': 50}, {'name': 'Strength', 'value': 'Full', 'price': 0}, {'name': 'Decaf', 'value': 'Normal', 'price': 0}, {'name': 'Temp', 'value': 'Normal', 'price': 0}, {'name': 'Honey', 'value': 'None', 'price': 0}, {'name': 'Syrup', 'value': 'None', 'price': 0}, {'name': 'White Sugar', 'value': '0', 'price': 0}, {'name': 'Raw Sugar', 'value': '0', 'price': 0}, {'name': 'Equal Sugar', 'value': '0', 'price': 0}, {'name': 'Extra shot', 'value': '0', 'price': 0}], 'price': 480, 'category': 'Hot Drinks'}"
1,11787,"{'name': 'Toastie', 'variant_name': 'Bacon, Egg and Cheese', 'variant_desc': '', 'variant_image': 'images/toastie-baconegg.png', 'options': [{'name': 'size', 'value': 'BACON-EGG-CHEESE', 'price': 800}, {'name': 'Bread', 'value': 'Multigrain', 'price': 0}], 'price': 800, 'category': 'Kitchen'}"
2,40606,"{'name': 'Flat White', 'variant_name': 'Extra Large', 'variant_desc': '', 'variant_image': 'images/coffee-flatwhite.png', 'options': [{'name': 'size', 'value': 'XLG', 'price': 540}, {'name': 'Milk', 'value': 'Lactose Free', 'price': 50}, {'name': 'Strength', 'value': 'Full', 'price': 0}, {'name': 'Decaf', 'value': 'Normal', 'price': 0}, {'name': 'Temp', 'value': 'Normal', 'price': 0}, {'name': 'Honey', 'value': 'None', 'price': 0}, {'name': 'Syrup', 'value': 'None', 'price': 0}, {'name': 'White Sugar', 'value': '0', 'price': 0}, {'name': 'Raw Sugar', 'value': '0', 'price': 0}, {'name': 'Equal Sugar', 'value': '0', 'price': 0}, {'name': 'Extra shot', 'value': '1', 'price': 50}], 'unitprice': 640, 'price': 640, 'category': 'Hot Drinks', 'quantity': 1}"
3,40606,"{'name': 'Latte', 'variant_name': 'Extra Large', 'variant_desc': '', 'variant_image': 'images/coffeecup-latte.png', 'options': [{'name': 'size', 'value': 'XLG', 'price': 540}, {'name': 'Milk', 'value': 'Lactose Free', 'price': 50}, {'name': 'Strength', 'value': 'Full', 'price': 0}, {'name': 'Decaf', 'value': 'Normal', 'price': 0}, {'name': 'Temp', 'value': 'Normal', 'price': 0}, {'name': 'Honey', 'value': 'None', 'price': 0}, {'name': 'Syrup', 'value': 'None', 'price': 0}, {'name': 'White Sugar', 'value': '0', 'price': 0}, {'name': 'Raw Sugar', 'value': '0', 'price': 0}, {'name': 'Equal Sugar', 'value': '0', 'price': 0}, {'name': 'Extra shot', 'value': '1', 'price': 50}], 'unitprice': 640, 'price': 640, 'category': 'Hot Drinks', 'quantity': 1}"
4,40488,"{'name': 'Cappuccino', 'variant_name': 'Extra Large', 'variant_desc': '', 'variant_image': 'images/coffeemug-cappuccino.png', 'options': [{'name': 'size', 'value': 'XLG', 'price': 540}, {'name': 'Milk', 'value': 'Oat Milk', 'price': 100}, {'name': 'Strength', 'value': 'Full', 'price': 0}, {'name': 'Decaf', 'value': 'Normal', 'price': 0}, {'name': 'Temp', 'value': 'Normal', 'price': 0}, {'name': 'Honey', 'value': 'None', 'price': 0}, {'name': 'Syrup', 'value': 'None', 'price': 0}, {'name': 'White Sugar', 'value': '0', 'price': 0}, {'name': 'Raw Sugar', 'value': '0', 'price': 0}, {'name': 'Equal Sugar', 'value': '0', 'price': 0}, {'name': 'Extra shot', 'value': '0', 'price': 0}], 'unitprice': 640, 'price': 640, 'category': 'Hot Drinks', 'quantity': 1}"


In [258]:
# Extract 'item', 'quantity', 'category', and 'price' fields from the 'cart' column
df['item'] = df['cart'].str['name']
df['quantity'] = df['cart'].apply(lambda x: x['quantity'] if 'quantity' in x else 1)
df['category'] = df['cart'].str['category']
df['price'] = df['cart'].str['price']

In [259]:
# Check if the fields were extracted correctly
df[['order_id', 'item', 'category', 'quantity', 'price']].head()

,order_id,item,category,quantity,price
0,11787,Latte,Hot Drinks,1,480
1,11787,Toastie,Kitchen,1,800
2,40606,Flat White,Hot Drinks,1,640
3,40606,Latte,Hot Drinks,1,640
4,40488,Cappuccino,Hot Drinks,1,640


In [260]:
# Print the unique categories of sold items
df['category'].unique()

array(['Hot Drinks', 'Kitchen', 'Cold Drinks', 'Food', None], dtype=object)

In [261]:
# Extract options details from 'cart' column
df['options'] = df['cart'].str['options']

In [262]:
df['options'][:3]

0              [{'name': 'size', 'value': 'LRG', 'price': 430}, {'name': 'Milk', 'value': 'Soy', 'price': 50}, {'name': 'Strength', 'value': 'Full', 'price': 0}, {'name': 'Decaf', 'value': 'Normal', 'price': 0}, {'name': 'Temp', 'value': 'Normal', 'price': 0}, {'name': 'Honey', 'value': 'None', 'price': 0}, {'name': 'Syrup', 'value': 'None', 'price': 0}, {'name': 'White Sugar', 'value': '0', 'price': 0}, {'name': 'Raw Sugar', 'value': '0', 'price': 0}, {'name': 'Equal Sugar', 'value': '0', 'price': 0}, {'name': 'Extra shot', 'value': '0', 'price': 0}]
1                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [263]:
# Convert options format
def extract_options_list(ops):
    if type(ops) == list:
        options_list = []
        for op in ops:
            options_list.append(
                [
                    op['name'],
                    op['value'],
                    op['price'],
                ]
            )
        return options_list
    else:
        return ops

df['options'] = df['options'].apply(extract_options_list)

In [264]:
df[['order_id', 'item', 'options', 'quantity', 'price']].head()

,order_id,item,options,quantity,price
0,11787,Latte,"[[size, LRG, 430], [Milk, Soy, 50], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 0, 0]]",1,480
1,11787,Toastie,"[[size, BACON-EGG-CHEESE, 800], [Bread, Multigrain, 0]]",1,800
2,40606,Flat White,"[[size, XLG, 540], [Milk, Lactose Free, 50], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 1, 50]]",1,640
3,40606,Latte,"[[size, XLG, 540], [Milk, Lactose Free, 50], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 1, 50]]",1,640
4,40488,Cappuccino,"[[size, XLG, 540], [Milk, Oat Milk, 100], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 0, 0]]",1,640


In [265]:
# Add 'item_tracking_id' to track the number of items in each order
df['item_tracking_id'] = (
    df.sort_values(['order_id','item'], ascending=[True, True])
      .groupby(['order_id'])
      .cumcount() + 1
)

In [266]:
df[['order_id', 'item', 'item_tracking_id', 'options']].head(10)

,order_id,item,item_tracking_id,options
0,11787,Latte,1,"[[size, LRG, 430], [Milk, Soy, 50], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 0, 0]]"
1,11787,Toastie,2,"[[size, BACON-EGG-CHEESE, 800], [Bread, Multigrain, 0]]"
2,40606,Flat White,1,"[[size, XLG, 540], [Milk, Lactose Free, 50], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 1, 50]]"
3,40606,Latte,2,"[[size, XLG, 540], [Milk, Lactose Free, 50], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 1, 50]]"
4,40488,Cappuccino,1,"[[size, XLG, 540], [Milk, Oat Milk, 100], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 0, 0]]"
5,40488,Flat White,2,"[[size, XLG, 540], [Milk, Lactose Free, 50], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 1, 50]]"
6,38252,Flat White,1,"[[size, XLG, 540], [Milk, Lactose Free, 50], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 1, 50]]"
7,38252,Latte,2,"[[size, XLG, 540], [Milk, Lactose Free, 50], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 1, 50]]"
8,13470,Mocha,1,"[[size, LRG, 480], [Milk, Full Cream, 0], [Strength, Full, 0], [Decaf, Normal, 0], [Temp, Normal, 0], [Honey, None, 0], [Syrup, None, 0], [White Sugar, 0, 0], [Raw Sugar, 0, 0], [Equal Sugar, 0, 0], [Extra shot, 0, 0]]"
9,13470,Thickshake (Syrup),2,"[[size, LRG, 800], [Flavour, Strawberry, 0]]"


In [267]:
# Separate elements in the `options` list into multiple rows
df = df.explode('options', ignore_index=True)

In [268]:
df[['order_id', 'item', 'options', 'quantity', 'price']].head()

,order_id,item,options,quantity,price
0,11787,Latte,"[size, LRG, 430]",1,480
1,11787,Latte,"[Milk, Soy, 50]",1,480
2,11787,Latte,"[Strength, Full, 0]",1,480
3,11787,Latte,"[Decaf, Normal, 0]",1,480
4,11787,Latte,"[Temp, Normal, 0]",1,480


In [269]:
# Extract option's name, value and price from 'options' column
df['option_name'] = df['options'].apply(lambda x: x[0] if type(x) == list else x)
df['option_value'] = df['options'].apply(lambda x: x[1] if type(x) == list else x)
df['option_price'] = df['options'].apply(lambda x: x[2] if type(x) == list else x)

In [270]:
df[[
    'order_id', 'item', 'options', 'option_name', 'option_value', 'option_price'
]].head()

,order_id,item,options,option_name,option_value,option_price
0,11787,Latte,"[size, LRG, 430]",size,LRG,430
1,11787,Latte,"[Milk, Soy, 50]",Milk,Soy,50
2,11787,Latte,"[Strength, Full, 0]",Strength,Full,0
3,11787,Latte,"[Decaf, Normal, 0]",Decaf,Normal,0
4,11787,Latte,"[Temp, Normal, 0]",Temp,Normal,0


In [271]:
# Extract 'size' column
df['size'] = df.apply(
    lambda x: x['option_value']
    if x['option_name'] == 'size'
    else None,
    axis=1,
)

In [272]:
# Extract 'unit_price' column
df['unit_price'] = df.apply(
    lambda x: x['option_price']
    if x['option_name'] == 'size'
    else None,
    axis=1,
)

In [273]:
# Create a new dataframe that contains size and unit_price values for each item per order
df_unitprice = (
    df[
        (~pd.isna(df['size'])) & (~pd.isna(df['unit_price']))
      ][
        ['order_id', 'item_tracking_id', 'size', 'unit_price']
    ]
)

df_unitprice.head()

,order_id,item_tracking_id,size,unit_price
0,11787,1,LRG,430.0
11,11787,2,BACON-EGG-CHEESE,800.0
13,40606,1,XLG,540.0
24,40606,2,XLG,540.0
35,40488,1,XLG,540.0


In [274]:
# Drop the rows that contain 'size' for 'option_name'
# in the original dataframe, and merge the two dataframes together 
df = df[df['option_name'] != 'size'].drop(
    ['size', 'unit_price'], axis=1
).merge(
    df_unitprice,
    how='left',
    on=['order_id', 'item_tracking_id'],
)

In [275]:
df.columns

Index(['order_id', 'customer_id', 'ip_addr', 'date_created', 'date_paid',
       'total', 'status', 'items', 'cart_size', 'cart_surcharge',
       'cart_total_price', 'cart_gst', 'cart_surcharge_display',
       'cart_total_price_display', 'cart_gst_display', 'cart', 'order_phone',
       'order_name', 'order_time', 'order_time_verbose', 'item', 'quantity',
       'category', 'price', 'options', 'item_tracking_id', 'option_name',
       'option_value', 'option_price', 'size', 'unit_price'],
      dtype='object')

In [276]:
df[
    [
        'order_id', 'item_tracking_id', 'item', 'option_name', 
        'option_value', 'option_price', 'size', 'unit_price'
    ]
].head()

,order_id,item_tracking_id,item,option_name,option_value,option_price,size,unit_price
0,11787,1,Latte,Milk,Soy,50,LRG,430.0
1,11787,1,Latte,Strength,Full,0,LRG,430.0
2,11787,1,Latte,Decaf,Normal,0,LRG,430.0
3,11787,1,Latte,Temp,Normal,0,LRG,430.0
4,11787,1,Latte,Honey,None,0,LRG,430.0


### Drop Irrelevant Columns and Clean Data

In [277]:
# Remove the columns containing duplicate or redundant information
df.drop(
    ['items', 'cart', 'options'], 
    axis=1, 
    inplace=True,
)

In [278]:
# Check the data type and unique values of each column
col_details = []
for col in df.columns:
    col_details.append(
        [
            col, 
            df[col].dtype,
            df[col].nunique(), 
            df[col].unique()[:10],
        ],
    )
    
pd.DataFrame(
    col_details,
    columns=[
        'Column Name', 
        'Data Type', 
        'Number of Unique Values', 
        'Unique Value Examples',
    ],
)

,Column Name,Data Type,Number of Unique Values,Unique Value Examples
0,order_id,Int64,35605,"[11787, 40606, 40488, 38252, 13470, 33377, 32604, 32496, 32399, 32177]"
1,customer_id,Int64,1546,"[517, 519, 776, 9, 1290, 12, 782, 20, 276, 791]"
2,ip_addr,object,16861,"[81741da2cc0e5207bfefbb4ec257b2393d5e13a0, ccf5e36bddfbce1a7de2c04874190a057b30588d, 1f8e636af94847a55418d2c03ccf4b1601d10512, d4c84c8e50cf80cc385d669000208e59e304fcca, 02d3475e227ed6eeb70daaa61d0ad1d0068dcd08, 7a1bb6402b347d07bc52ab100f2b67e7e027a106, 3cba56ec1336930ecabe05c12ff862a3ceb12fe3, b756854eea913de767c8ab7df7b7b6329cdf23cc, e82aac3f7a4536d950da2cbe0a0fa3bf389dc16f, 6c12bae969d1a308b71e8fd1e883d57fd32465ef]"
3,date_created,datetime64[ns],35580,"[2021-04-14T05:56:06.000000000, 2024-03-05T10:16:40.000000000, 2024-03-01T11:40:20.000000000, 2023-12-05T08:48:02.000000000, 2021-06-20T12:10:17.000000000, 2023-07-04T10:30:31.000000000, 2023-06-09T08:21:57.000000000, 2023-06-05T11:30:51.000000000, 2023-06-02T11:08:35.000000000, 2023-05-26T11:58:13.000000000]"
4,date_paid,datetime64[ns],35586,"[2021-04-14T05:56:37.000000000, 2024-03-05T10:17:23.000000000, 2024-03-01T11:40:58.000000000, 2023-12-05T08:49:24.000000000, 2021-06-20T12:11:09.000000000, 2023-07-04T10:30:48.000000000, 2023-06-09T08:22:21.000000000, 2023-06-05T11:31:15.000000000, 2023-06-02T11:08:56.000000000, 2023-05-26T11:58:56.000000000]"
5,total,Int64,748,"[1280, 2560, 2816, 3840, 5120, 6400, 770, 2050, 3330, 4610]"
6,status,Int64,3,"[2, 1, 10]"
7,cart_size,int64,15,"[2, 4, 3, 1, 5, 6, 8, 10, 7, 9]"
8,cart_surcharge,float64,139,"[0.0, 256.0, 17.0, 140.0, 47.0, 187.0, 94.0, 420.0, 141.0, 48.0]"
9,cart_total_price,int64,748,"[1280, 2560, 2816, 3840, 5120, 6400, 770, 2050, 3330, 4610]"


In [279]:
df[
    ['date_created', 'date_paid', 'order_time', 'order_time_verbose']
].drop_duplicates()[:5]

,date_created,date_paid,order_time,order_time_verbose
0,2021-04-14 05:56:06,2021-04-14 05:56:37,1618344900,6:15am
11,2024-03-05 10:16:40,2024-03-05 10:17:23,ASAP,ASAP
31,2024-03-01 11:40:20,2024-03-01 11:40:58,ASAP,ASAP
51,2023-12-05 08:48:02,2023-12-05 08:49:24,ASAP,ASAP
71,2021-06-20 12:10:17,2021-06-20 12:11:09,ASAP,ASAP


Let's convert `order_time` column to *datetime* format and replace `ASAP` values with the corresponding `date_paid` values.

In [280]:
df['order_time_'] = df.apply(
    lambda x: datetime.fromtimestamp(int(x['order_time']))
    if x['order_time'] != 'ASAP' 
    else x['date_paid'],
    axis=1,
)

In [281]:
df[
    ['date_created', 'date_paid', 'order_time', 
     'order_time_', 'order_time_verbose']
].drop_duplicates()[:5]

,date_created,date_paid,order_time,order_time_,order_time_verbose
0,2021-04-14 05:56:06,2021-04-14 05:56:37,1618344900,2021-04-14 06:15:00,6:15am
11,2024-03-05 10:16:40,2024-03-05 10:17:23,ASAP,2024-03-05 10:17:23,ASAP
31,2024-03-01 11:40:20,2024-03-01 11:40:58,ASAP,2024-03-01 11:40:58,ASAP
51,2023-12-05 08:48:02,2023-12-05 08:49:24,ASAP,2023-12-05 08:49:24,ASAP
71,2021-06-20 12:10:17,2021-06-20 12:11:09,ASAP,2021-06-20 12:11:09,ASAP


The other time columns including `date_created`, `date_paid`, `order_time`, `order_time_verbose` are redundant and should be dumped.

In [282]:
# Drop the redundant time columns
df.drop(
    [
        'date_created',
        'date_paid',
        'order_time',
        'order_time_verbose',
    ],
    axis=1,
    inplace=True,
)

In [283]:
# Rename 'order_time_' to 'order_time'
df.rename(
    columns={'order_time_': 'order_time'}, inplace=True
)

In [284]:
# Investigate the '_display' columns and their corresponding numeric columns
df[
    ['cart_gst', 'cart_gst_display',
     'cart_surcharge', 'cart_surcharge_display',
     'cart_total_price', 'cart_total_price_display']
].drop_duplicates().head()

,cart_gst,cart_gst_display,cart_surcharge,cart_surcharge_display,cart_total_price,cart_total_price_display
0,116.363636,$1.16,0.0,$0.00,1280,$12.80
1881,232.727273,$2.33,0.0,$0.00,2560,$25.60
2480,256.000000,$2.56,256.0,$2.56,2816,$28.16
2501,349.090909,$3.49,0.0,$0.00,3840,$38.40
2603,465.454545,$4.65,0.0,$0.00,5120,$51.20


The `cart_gst_display`, `cart_surcharge_display`, and `cart_total_price_display` columns represent the string-formatted prices of the corresponding numeric values in the `cart_gst`, `cart_surcharge`, `cart_total_price` columns. Therefore, those `_display` columns should be removed, and the numeric values should be divided by 100 to accurately reflect the true prices.

In [285]:
df.drop(
    [
        'cart_surcharge_display', 
        'cart_total_price_display', 
        'cart_gst_display',
    ],
    axis=1, 
    inplace=True,
)

In [286]:
df.select_dtypes(include='number').drop_duplicates().head()

,order_id,customer_id,total,status,cart_size,cart_surcharge,cart_total_price,cart_gst,quantity,price,item_tracking_id,option_price,unit_price
0,11787,517,1280,2,2,0.0,1280,116.363636,1,480,1,50,430.0
1,11787,517,1280,2,2,0.0,1280,116.363636,1,480,1,0,430.0
10,11787,517,1280,2,2,0.0,1280,116.363636,1,800,2,0,800.0
11,40606,519,1280,2,2,0.0,1280,116.363636,1,640,1,50,540.0
12,40606,519,1280,2,2,0.0,1280,116.363636,1,640,1,0,540.0


Divide the quantitative pricing columns, including `total`, `cart_surcharge`, `cart_total_price`, `cart_gst`, `price`, and `option_price`, by 100 to accurately reflect the true prices.

In [287]:
cols_to_adjust = [
    'total', 'cart_surcharge', 'cart_total_price', 
    'cart_gst', 'price', 'option_price', 'unit_price'
]

for col in cols_to_adjust:
    df[col] = df[col] / 100

In [288]:
df[cols_to_adjust].drop_duplicates().head()

,total,cart_surcharge,cart_total_price,cart_gst,price,option_price,unit_price
0,12.8,0.0,12.8,1.163636,4.8,0.5,4.3
1,12.8,0.0,12.8,1.163636,4.8,0.0,4.3
10,12.8,0.0,12.8,1.163636,8.0,0.0,8.0
11,12.8,0.0,12.8,1.163636,6.4,0.5,5.4
12,12.8,0.0,12.8,1.163636,6.4,0.0,5.4


Since the `cart_size` and `cart_gst` columns can be derived through aggregation, and `total` is the same as `cart_total_price`, we can safely remove the three fields from the dataframe to reduce redundancy.

In [289]:
# Drop 'cart_size', 'cart_gst' and 'total'
df.drop(
    [
        'cart_size',
        'cart_gst',
        'total',
    ], 
    axis=1, 
    inplace=True,
)

In [290]:
# Rename 'cart_total_price' and 'price' appropriately
df.rename(
    columns={'cart_total_price': 'order_price', 'price': 'item_price'},
    inplace=True,
)

The fields like `ip_addr`, `order_phone`, and `order_name` have little relevance to the analysis and can be removed as well to streamline the dataset.

In [291]:
df.drop(
    [
        'ip_addr', 
        'order_phone',
        'order_name',
    ], 
    axis=1, 
    inplace=True,
)

In [292]:
df.columns

Index(['order_id', 'customer_id', 'status', 'cart_surcharge', 'order_price',
       'item', 'quantity', 'category', 'item_price', 'item_tracking_id',
       'option_name', 'option_value', 'option_price', 'size', 'unit_price',
       'order_time'],
      dtype='object')

### Impute NULL values

In [293]:
# Identify the columns with NULl values
cols_with_null = []
for col in df.columns:
    if any(pd.isna(x) for x in df[col]):
        cols_with_null.append(col)

In [294]:
cols_with_null

['cart_surcharge', 'category']

#### Fill in the missing values for the `category` field

In [295]:
# Print the unique values of category
df['category'].unique()

array(['Hot Drinks', 'Kitchen', 'Cold Drinks', 'Food', None], dtype=object)

In [296]:
# Find the items that have multiple categories
df_ = (
    df[['item', 'category']][~pd.isna(df['category'])]
    .drop_duplicates().groupby('item').count()
)
items_multiple_cat = list(df_[df_['category'] > 1].index)
items_multiple_cat

['Croissant', 'Toasted Roll', 'Toastie']

In [297]:
# Check what categories the target items have been assigned to
df[
    (df['item'].isin(items_multiple_cat)) & (~pd.isna(df['category']))
][['item', 'category']].drop_duplicates().sort_values('item')

,item,category
1555,Croissant,Kitchen
2075,Croissant,Food
675,Toasted Roll,Kitchen
1128,Toasted Roll,Food
10,Toastie,Kitchen
1511,Toastie,Food


Let's resolve the conflicts using the most common category for each target item.

In [298]:
for item in items_multiple_cat:
    most_common_cat = df[df['item'] == item]['category'].value_counts().index[0]
    df.loc[
        df[df['item'] == item].index, 'category'
    ] = most_common_cat

In [299]:
# Check if the conflicts have been resolved
df[
    (df['item'].isin(items_multiple_cat)) & (~pd.isna(df['category']))
][['item', 'category']].drop_duplicates().sort_values('item')

,item,category
1555,Croissant,Kitchen
675,Toasted Roll,Kitchen
10,Toastie,Kitchen


In [300]:
# Print the unique values of category
df['category'].unique()

array(['Hot Drinks', 'Kitchen', 'Cold Drinks', None], dtype=object)

Now the sold items are only categorised into `Hot Drinks`, `Kitchen` and `Cold Drinks`.

In [301]:
# Find the list of items where the 'category' field is missing
items_missing_cat = list(sorted(df[pd.isna(df['category'])]['item'].unique()))
items_missing_cat

['Affogato',
 'Babychino',
 'Cappuccino',
 'Chai Latte',
 'Dirty Chai Latte',
 'Espresso',
 'Flat White',
 'Freshly Squeezed Juice',
 'Golden Latte',
 'Hot Chocolate',
 'Latte',
 'Long Black',
 'Long Macchiato',
 'Milkshake (Syrup)',
 'Mocha',
 'Piccolo',
 'Short Black',
 'Smoothie',
 'Toasties',
 'White Hot Chocolate']

In [302]:
# Create an dictionary with items as keys and categories as values
df_item_cat = (
    df[~pd.isna(df['category'])]
    [['item', 'category']].drop_duplicates().sort_values('item')
)
dict_item_cat = {x[0]: x[1] for x in df_item_cat.to_numpy()}
dict_item_cat

{'(Entree) Pasta': 'Kitchen',
 '(Main) Pasta': 'Kitchen',
 'Acai Smoothie bowl (VO)': 'Kitchen',
 'Affogato': 'Cold Drinks',
 'Babychino': 'Hot Drinks',
 'Big Breakfast': 'Kitchen',
 'Build Your Own Breakfast': 'Kitchen',
 'Cappuccino': 'Hot Drinks',
 'Chai Latte': 'Hot Drinks',
 'Coffee Frappe': 'Cold Drinks',
 'Coffee Milkshake': 'Cold Drinks',
 'Croissant': 'Kitchen',
 'Deep Fryer': 'Kitchen',
 'Dirty Chai Latte': 'Hot Drinks',
 'Eggs Benedict': 'Kitchen',
 'Eggs Benny': 'Kitchen',
 'Eggs Florentine': 'Kitchen',
 'Eggs and Bacon': 'Kitchen',
 'Espresso': 'Hot Drinks',
 'Flat White': 'Hot Drinks',
 'French Toast with Streaky Bacon': 'Kitchen',
 'Freshly Squeezed Juice': 'Cold Drinks',
 'Golden Latte': 'Hot Drinks',
 'Hot Chocolate': 'Hot Drinks',
 'Hot Potatoes': 'Kitchen',
 'Ice Chai': 'Cold Drinks',
 'Ice Latte': 'Cold Drinks',
 'Iced Chocolate with Cream': 'Cold Drinks',
 'Iced Coffee with Cream': 'Cold Drinks',
 'Iced Mocha with Cream': 'Cold Drinks',
 'Kids Crepes': 'Kitchen',
 

In [303]:
# Find the items that are still not classified
items_not_classified = [
    item for item in items_missing_cat if item not in dict_item_cat
]
items_not_classified

['Short Black', 'Toasties']

In [304]:
# View the option_name and option_value associated with the unclassified items
df[df['item'].isin(items_not_classified)][
    ['item', 'option_name', 'option_value']
].sort_values('item')

,item,option_name,option_value
178006,Short Black,Milk,Full Cream
178007,Short Black,Strength,Full
178008,Short Black,Honey,None
178009,Short Black,Syrup,None
178010,Short Black,White Sugar,0
178011,Short Black,Raw Sugar,0
178012,Short Black,Extra shot,0
49001,Toasties,Bread,White
352501,Toasties,Bread,White
352502,Toasties,Bread,Sourdough


Since `Toasties` and `Toastie` refer to the same item, they should both be classified under `Kitchen`. Given that the `Short Black` item shares similar options with other `Hot Drinks` items, it should be classified under `Hot Drinks`.

In [305]:
df['item'] = df['item'].apply(
    lambda x: 'Toastie' if x == 'Toasties' else x
)

In [306]:
# Reflect the re-categorisation of 'Short Black' in dict_item_cat
dict_item_cat['Short Black'] = 'Hot Drinks'

In [307]:
def impute_category(x):
    if pd.isna(x['category']):
        return dict_item_cat[x['item']]
    else:
        return x['category']
    
df['category'] = df.apply(impute_category, axis=1)

In [308]:
# Check if all items have been categorised
if len(df[pd.isna(df['category'])]) == 0:
    print('All items have been categorised')
else:
    print(
        'Some items remain uncategorized. '
        'Please investigate further to ensure '
        'all items are properly classified.'
    )

All items have been categorised


Before we proceed to impute the `cart_surcharge` column, let's take a moment to review the items classified as `Hot Drinks`, `Cold Drinks`, and `Kitchen`.

In [309]:
df['category'].unique()

array(['Hot Drinks', 'Kitchen', 'Cold Drinks'], dtype=object)

In [310]:
pd.DataFrame(
    {
        'category': df['category'].unique(),
        'item': [
            df[df['category'] == cat]['item'].unique() 
            for cat in df['category'].unique()
        ],
    }
)

,category,item
0,Hot Drinks,"[Latte, Flat White, Cappuccino, Mocha, Babychino, Hot Chocolate, Chai Latte, Long Black, White Hot Chocolate, Macha Latte, Magic, Dirty Chai Latte, Golden Latte, Piccolo, Espresso, Long Macchiato, Short Macchiato, Short Black]"
1,Kitchen,"[Toastie, Toasted Roll, Kids Pancakes, Croissant, French Toast with Streaky Bacon, Build Your Own Breakfast, Eggs Benedict, (Main) Pasta, Kids Crepes, Eggs and Bacon, Big Breakfast, Hot Potatoes, (Entree) Pasta, Pork Benny, Acai Smoothie bowl (VO), Eggs Benny, Eggs Florentine, Plain Toast, Deep Fryer]"
2,Cold Drinks,"[Thickshake (Syrup), Smoothie, Ice Latte, Iced Coffee with Cream, Iced Mocha with Cream, Ice Chai, Iced Chocolate with Cream, Coffee Frappe, Milkshake (Syrup), Freshly Squeezed Juice, Muffin, Coffee Milkshake, Affogato]"


The `Muffin` item is currently classified as a `Cold Drink`, which seems not sensible...

In [311]:
# Dive in to investigate the 'Muffin' item
df[df['item'] == 'Muffin'][
    ['item', 'category', 'option_name', 'option_value', 'size', 'unit_price']
].drop_duplicates()

,item,category,option_name,option_value,size,unit_price
2056,Muffin,Cold Drinks,Heated,No,ORANGE-POPPYSEED,3.9
2173,Muffin,Cold Drinks,Heated,No,CHOC-MUD,3.9
2281,Muffin,Cold Drinks,Heated,Yes,ORANGE-POPPYSEED,3.9
2282,Muffin,Cold Drinks,Heated,Yes,APPLE-CINNAMON,3.9
2666,Muffin,Cold Drinks,Heated,No,RASPBERRY-WHITECHOC,3.9
3990,Muffin,Cold Drinks,Heated,Yes,CHOC-MUD,3.9
4276,Muffin,Cold Drinks,Heated,Yes,RASPBERRY-WHITECHOC,3.9
4433,Muffin,Cold Drinks,Heated,No,APPLE-CINNAMON,3.9
4593,Muffin,Cold Drinks,Heated,No,RASPBERRY-WHITECHOC,4.5
7011,Muffin,Cold Drinks,Heated,Yes,APPLE-CINNAMON,4.5


The `Muffin` item has been misclassified. We should be reclassify it under `Kitchen`.

In [312]:
# Reclassify 'Muffin' items under 'Kitchen'
df.loc[
    df[df['item'] == 'Muffin'].index, 'category'
] = 'Kitchen'

In [313]:
df[df['item'] == 'Muffin']['category'].unique()

array(['Kitchen'], dtype=object)

#### Fill in the missing values for the `cart_surcharge` field

In [314]:
# Check if the 'cart_surcharge' field has missing values
any(pd.isna(v) for v in df['cart_surcharge'].unique())

True

In [315]:
# Let's evaluate the relationship between 'cart_surcharge' and 'order_price'
df_ = df[['cart_surcharge', 'order_price']][~pd.isna(df['cart_surcharge'])].copy()
df_['factor'] = df_['cart_surcharge'] / df_['order_price']

In [316]:
df_['factor'].value_counts()

0.000000    494261
0.090909      3188
0.090909      1159
0.090909       482
0.016553        14
0.016544         9
0.016204         2
Name: factor, dtype: int64

In [317]:
df_['factor'].value_counts()[0] / df_['factor'].value_counts().sum()

0.9902747863718783

Let's apply the most frequent factor, 0 (99%), to calculate the unkown `cart_surcharge`.

In [318]:
# Set the missing value in 'cart_surcharge' column to 0 
df['cart_surcharge'] = df['cart_surcharge'].apply(
    lambda x: 0 if pd.isna(x) else x
)

In [319]:
# Check if the missing values in 'cart_surcharge' are all filled
any(pd.isna(i) for i in df['cart_surcharge'].unique())

False

In [320]:
df.head()

,order_id,customer_id,status,cart_surcharge,order_price,item,quantity,category,item_price,item_tracking_id,option_name,option_value,option_price,size,unit_price,order_time
0,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Milk,Soy,0.5,LRG,4.3,2021-04-14 06:15:00
1,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Strength,Full,0.0,LRG,4.3,2021-04-14 06:15:00
2,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Decaf,Normal,0.0,LRG,4.3,2021-04-14 06:15:00
3,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Temp,Normal,0.0,LRG,4.3,2021-04-14 06:15:00
4,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Honey,None,0.0,LRG,4.3,2021-04-14 06:15:00


### Add a Customer Churn Label

In [335]:
# Create a new dataframe for labeling churns
df_churn = df[
    ['customer_id', 'order_time', 'order_id']
].drop_duplicates().sort_values(
    ['customer_id', 'order_time', 'order_id'],
    ascending=[True, True, True],
).reset_index(drop=True).copy()

In [336]:
# Rename 'order_time' to 'prev_order_time'
df_churn.rename(
    columns={'order_time': 'prev_order_time'},
    inplace=True
)

df_churn.head()

,customer_id,prev_order_time,order_id
0,4,2019-05-05 22:20:05,3272
1,4,2019-05-06 06:00:00,3271
2,4,2019-05-06 06:00:00,3273
3,4,2019-05-06 06:15:00,3270
4,5,2019-05-06 11:00:02,3278


In [337]:
# Create a new column 'next_order_time' 
# for calculating purchase intervals
df_churn['next_order_time'] = df_churn.sort_values(
    by=['customer_id', 'prev_order_time'], 
    ascending=[True, True],
).groupby(['customer_id'])['prev_order_time'].shift(-1)

df_churn.head()

,customer_id,prev_order_time,order_id,next_order_time
0,4,2019-05-05 22:20:05,3272,2019-05-06 06:00:00
1,4,2019-05-06 06:00:00,3271,2019-05-06 06:00:00
2,4,2019-05-06 06:00:00,3273,2019-05-06 06:15:00
3,4,2019-05-06 06:15:00,3270,NaT
4,5,2019-05-06 11:00:02,3278,2019-05-06 11:01:43


In [338]:
# Create new column for recording purchasing interval in days
def get_interval(x):
    if pd.isna(x['next_order_time']):
        return None
    else:
        return (
            x['next_order_time'].date()
            - x['prev_order_time'].date()
        ).days

df_churn['purchase_interval_days'] = (
    df_churn.apply(
        lambda x: get_interval(x), axis=1
    )
)

df_churn.head()

,customer_id,prev_order_time,order_id,next_order_time,purchase_interval_days
0,4,2019-05-05 22:20:05,3272,2019-05-06 06:00:00,1.0
1,4,2019-05-06 06:00:00,3271,2019-05-06 06:00:00,0.0
2,4,2019-05-06 06:00:00,3273,2019-05-06 06:15:00,0.0
3,4,2019-05-06 06:15:00,3270,NaT,NaN
4,5,2019-05-06 11:00:02,3278,2019-05-06 11:01:43,0.0


In [339]:
# Calculate the standard deviation of purchasing 
# interval days for each customer
df_churn['interval_std'] = (
    df_churn.groupby('customer_id')
    ['purchase_interval_days'].transform(np.std)
)

df_churn.head()

,customer_id,prev_order_time,order_id,next_order_time,purchase_interval_days,interval_std
0,4,2019-05-05 22:20:05,3272,2019-05-06 06:00:00,1.0,0.577350
1,4,2019-05-06 06:00:00,3271,2019-05-06 06:00:00,0.0,0.577350
2,4,2019-05-06 06:00:00,3273,2019-05-06 06:15:00,0.0,0.577350
3,4,2019-05-06 06:15:00,3270,NaT,NaN,0.577350
4,5,2019-05-06 11:00:02,3278,2019-05-06 11:01:43,0.0,60.412166


In [340]:
# Calculate the churn threshold as two times the standard 
# deviation of the intervals per customer
df_churn['churn_threshold'] = df_churn['interval_std'] * 2
df_churn.head()

,customer_id,prev_order_time,order_id,next_order_time,purchase_interval_days,interval_std,churn_threshold
0,4,2019-05-05 22:20:05,3272,2019-05-06 06:00:00,1.0,0.577350,1.154701
1,4,2019-05-06 06:00:00,3271,2019-05-06 06:00:00,0.0,0.577350,1.154701
2,4,2019-05-06 06:00:00,3273,2019-05-06 06:15:00,0.0,0.577350,1.154701
3,4,2019-05-06 06:15:00,3270,NaT,NaN,0.577350,1.154701
4,5,2019-05-06 11:00:02,3278,2019-05-06 11:01:43,0.0,60.412166,120.824331


In [341]:
# Create a new dataframe that maps the churn threshold 
# to each customer
df_customer_churn = df_churn[
    ['customer_id', 'churn_threshold']
].drop_duplicates().reset_index(drop=True)

print(df_customer_churn.shape)
print(df_customer_churn.info())
df_customer_churn.head()

(1546, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1546 entries, 0 to 1545
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      1546 non-null   Int64  
 1   churn_threshold  673 non-null    float64
dtypes: Int64(1), float64(1)
memory usage: 25.8 KB
None


,customer_id,churn_threshold
0,4,1.154701
1,5,120.824331
2,6,2158.089896
3,7,NaN
4,8,60.411908


For customers with NaN values in the `churn_threshold`, it likely indicates they have only placed one order. In this case, their churn threshold can be set as twice the standard deviation of purchase intervals across all customers.

In [347]:
churn_threshold_all = np.std(df_churn['purchase_interval_days']) * 2
print(churn_threshold_all)

106.95824075222875


In [350]:
# Impute NaN in 'churn_threshold' with two times the standard 
# deviation of the intervals across all customers
df_customer_churn['churn_threshold'] = (
    df_customer_churn['churn_threshold'].apply(
        lambda x: churn_threshold_all if pd.isna(x) else x
    )
)

df_customer_churn.head()

,customer_id,churn_threshold
0,4,1.154701
1,5,120.824331
2,6,2158.089896
3,7,106.958241
4,8,60.411908


In [353]:
df = df.merge(
    df_customer_churn,
    how='left',
    on='customer_id',
)

In [354]:
df.columns

Index(['order_id', 'customer_id', 'status', 'cart_surcharge', 'order_price',
       'item', 'quantity', 'category', 'item_price', 'item_tracking_id',
       'option_name', 'option_value', 'option_price', 'size', 'unit_price',
       'order_time', 'churn_threshold'],
      dtype='object')

In [355]:
df.head()

,order_id,customer_id,status,cart_surcharge,order_price,item,quantity,category,item_price,item_tracking_id,option_name,option_value,option_price,size,unit_price,order_time,churn_threshold
0,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Milk,Soy,0.5,LRG,4.3,2021-04-14 06:15:00,44.242406
1,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Strength,Full,0.0,LRG,4.3,2021-04-14 06:15:00,44.242406
2,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Decaf,Normal,0.0,LRG,4.3,2021-04-14 06:15:00,44.242406
3,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Temp,Normal,0.0,LRG,4.3,2021-04-14 06:15:00,44.242406
4,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Honey,None,0.0,LRG,4.3,2021-04-14 06:15:00,44.242406


### Export the Cleaned Data to a CSV file

In [356]:
df.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_full_mx.csv', 
    header=True, 
    index=False,
)

<hr>

## Split the DataFrame Based on the Three Item Categories

In [636]:
# Split the dataframe based on the 3 categories of sold items
df_hotdrinks = df[df['category'] == 'Hot Drinks'].copy()
df_colddrinks = df[df['category'] == 'Cold Drinks'].copy()
df_kitchen = df[df['category'] == 'Kitchen'].copy()

### Transform Hot Drinks Data for Better Usability

In [637]:
df_hotdrinks.head()

,order_id,customer_id,status,cart_surcharge,order_price,item,quantity,category,item_price,item_tracking_id,option_name,option_value,option_price,size,unit_price,order_time
0,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Milk,Soy,0.5,LRG,4.3,2021-04-14 06:15:00
1,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Strength,Full,0.0,LRG,4.3,2021-04-14 06:15:00
2,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Decaf,Normal,0.0,LRG,4.3,2021-04-14 06:15:00
3,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Temp,Normal,0.0,LRG,4.3,2021-04-14 06:15:00
4,11787,517,2,0.0,12.8,Latte,1,Hot Drinks,4.8,1,Honey,None,0.0,LRG,4.3,2021-04-14 06:15:00


#### Pivot `option_name` into headers

In [638]:
# Pivot the 'option_name' column into headers, 
# using the 'option_value' column for their corresponding values
df_hotdrinks_pivoted = df_hotdrinks.pivot(
    index=['order_id', 'item_tracking_id'], 
    columns='option_name', 
    values='option_value',
).reset_index(drop=False)

In [639]:
# Check the pivoted table schema
print(df_hotdrinks_pivoted.shape)
print(df_hotdrinks_pivoted.columns)
df_hotdrinks_pivoted.head()

(52324, 12)
Index(['order_id', 'item_tracking_id', 'Decaf', 'Equal Sugar', 'Extra shot',
       'Honey', 'Milk', 'Raw Sugar', 'Strength', 'Syrup', 'Temp',
       'White Sugar'],
      dtype='object', name='option_name')


option_name,order_id,item_tracking_id,Decaf,Equal Sugar,Extra shot,Honey,Milk,Raw Sugar,Strength,Syrup,Temp,White Sugar
0,3270,1,NaN,NaN,0,None,Full Cream,0,Full,None,NaN,0
1,3271,1,NaN,NaN,0,None,Full Cream,0,Full,None,NaN,0
2,3272,1,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0
3,3273,1,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0
4,3274,1,NaN,0,0,None,Full Cream,0,Full,None,NaN,0


In [640]:
# Gather other attributes for each item per order
df_hotdrinks_attr = df_hotdrinks.drop(
    ['option_name', 'option_value', 'option_price'], 
    axis=1,
).drop_duplicates()

# Join the two tables to create a final table for 
# hot drinks, ensuring each row represents one item per order
df_hotdrinks_final = df_hotdrinks_attr.merge(
    df_hotdrinks_pivoted, how='inner', on=['order_id', 'item_tracking_id']
)

#### Create a new `option_price` column

In [641]:
# Create a new 'option_price' column that represents 
# the difference between 'item_price' per unit and 'unit_price'
df_hotdrinks_final['option_price'] = (
    df_hotdrinks_final['item_price'] / df_hotdrinks_final['quantity'] 
    - df_hotdrinks_final['unit_price']
)

In [642]:
# Find the order that includes the highest number of hot drinks
df_hotdrinks_final['order_id'].value_counts()[:1]

17135    13
Name: order_id, dtype: Int64

In [643]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 17135
df_hotdrinks_final[df_hotdrinks_final['order_id'] == 17135][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
]

,order_id,item_tracking_id,item,size,quantity,order_price,item_price,unit_price,option_price
16199,17135,1,Cappuccino,REG,1,59.7,4.2,3.7,0.5
16200,17135,10,Latte,LRG,1,59.7,4.8,4.3,0.5
16201,17135,2,Cappuccino,LRG,1,59.7,4.3,4.3,0.0
16202,17135,4,Chai Latte,LRG,1,59.7,4.8,4.8,0.0
16203,17135,5,Chai Latte,LRG,1,59.7,5.5,4.8,0.7
16204,17135,13,Mocha,REG,1,59.7,4.0,4.0,0.0
16205,17135,7,Hot Chocolate,LRG,1,59.7,4.8,4.8,0.0
16206,17135,8,Hot Chocolate,LRG,1,59.7,4.8,4.8,0.0
16207,17135,3,Cappuccino,LRG,1,59.7,4.3,4.3,0.0
16208,17135,11,Latte,LRG,1,59.7,4.3,4.3,0.0


Now, let's clean the option columns, including `Decaf`, `Equal Sugar`, `Extra shot`, `Honey`, `Milk`, `Raw Sugar`, `Strength`, `Syrup`, `Temp`, and `White Sugar`.

#### Impute missing options

In [644]:
option_columns = list(df_hotdrinks['option_name'].unique())
option_columns

['Milk',
 'Strength',
 'Decaf',
 'Temp',
 'Honey',
 'Syrup',
 'White Sugar',
 'Raw Sugar',
 'Equal Sugar',
 'Extra shot']

In [645]:
def show_unique_options(df, option_columns):
    unique_options = []
    for col in option_columns:
        unique_options.append(
            list(df[col].unique())
        )

    return pd.DataFrame(
        {
            'Options': option_columns,
            'Unique Values': unique_options,
        }
    )

In [646]:
show_unique_options(df_hotdrinks_final, option_columns)

,Options,Unique Values
0,Milk,"[Soy, Lactose Free, Oat Milk, Full Cream, Skim, Bonsoy, Almond, Almond Milk Lab, Coconut Milk, Happy Soy Boy, Happy Happy Soy Boy, nan]"
1,Strength,"[Full, nan, Half, Extra Strong, Quarter]"
2,Decaf,"[Normal, nan, Decaf]"
3,Temp,"[Normal, Hot, nan, Warm]"
4,Honey,"[None, nan, Honey]"
5,Syrup,"[None, nan, Vanilla, Caramel, Hazelnut, Butterscotch, Irish Cream]"
6,White Sugar,"[0, 0.5, 1, nan, 3, 2, 1.5, 2.5, 4]"
7,Raw Sugar,"[0, 1, nan, 1.5, 2, 0.5, 3, 2.5, 4, 3.5]"
8,Equal Sugar,"[0, nan, 0.5, 2, 1, 1.5, 4, 3]"
9,Extra shot,"[0, 1, nan, 2, 5, 3]"


In [647]:
def impute_options(df, cols, val):
    for col in cols:
        df[col] = (
            df[col].apply(
                lambda x: val if pd.isna(x) else x
            )
        )

In [648]:
# Impute the nan in 'Milk',' Honey', and 'Syrup' 
# options with 'None'
impute_options(
    df=df_hotdrinks_final,
    cols=['Milk', 'Honey', 'Syrup'],
    val='None',
)

In [649]:
# Impute the nan in 'White Sugar', 'Raw Sugar', 
# 'Equal Sugar' and 'Extra shot' options with '0'
impute_options(
    df=df_hotdrinks_final,
    cols=[
        'White Sugar', 'Raw Sugar', 'Equal Sugar', 'Extra shot'
    ],
    val='0',
)

In [650]:
# Impute the missing values in 'Strength', 'Decaf' 
# and 'Temp' with the most frequent choices
for col in ['Strength', 'Decaf', 'Temp']:
    most_common = df_hotdrinks_final[col].value_counts().index[0]
    df_hotdrinks_final[col] = (
        df_hotdrinks_final[col].apply(
            lambda x: most_common if pd.isna(x) else x
        )
    )

In [651]:
show_unique_options(df_hotdrinks_final, option_columns)

,Options,Unique Values
0,Milk,"[Soy, Lactose Free, Oat Milk, Full Cream, Skim, Bonsoy, Almond, Almond Milk Lab, Coconut Milk, Happy Soy Boy, Happy Happy Soy Boy, None]"
1,Strength,"[Full, Half, Extra Strong, Quarter]"
2,Decaf,"[Normal, Decaf]"
3,Temp,"[Normal, Hot, Warm]"
4,Honey,"[None, Honey]"
5,Syrup,"[None, Vanilla, Caramel, Hazelnut, Butterscotch, Irish Cream]"
6,White Sugar,"[0, 0.5, 1, 3, 2, 1.5, 2.5, 4]"
7,Raw Sugar,"[0, 1, 1.5, 2, 0.5, 3, 2.5, 4, 3.5]"
8,Equal Sugar,"[0, 0.5, 2, 1, 1.5, 4, 3]"
9,Extra shot,"[0, 1, 2, 5, 3]"


All missing values in the option columns have now been filled in appropriately.

In [652]:
print(df_hotdrinks_final.shape)
print(df_hotdrinks_final.info())

(52324, 24)
<class 'pandas.core.frame.DataFrame'>
Int64Index: 52324 entries, 0 to 52323
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          52324 non-null  Int64         
 1   customer_id       52324 non-null  Int64         
 2   status            52324 non-null  Int64         
 3   cart_surcharge    52324 non-null  float64       
 4   order_price       52324 non-null  float64       
 5   item              52324 non-null  object        
 6   quantity          52324 non-null  int64         
 7   category          52324 non-null  object        
 8   item_price        52324 non-null  float64       
 9   item_tracking_id  52324 non-null  int64         
 10  size              52324 non-null  object        
 11  unit_price        52324 non-null  float64       
 12  order_time        52324 non-null  datetime64[ns]
 13  Decaf             52324 non-null  object        
 14  Equal Suga

#### Export the Hot Drinks data as a separate CSV file

In [653]:
df_hotdrinks_final.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_hotdrinks_pivoted_mx.csv',
    header=True,
    index=False,
)

### Transform Cold Drinks Data for Better Usability

In [654]:
df_colddrinks.reset_index(drop=True, inplace=True)
print(df_colddrinks.shape)
print(df_colddrinks.info())

(13248, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13248 entries, 0 to 13247
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          13248 non-null  Int64         
 1   customer_id       13248 non-null  Int64         
 2   status            13248 non-null  Int64         
 3   cart_surcharge    13248 non-null  float64       
 4   order_price       13248 non-null  float64       
 5   item              13248 non-null  object        
 6   quantity          13248 non-null  int64         
 7   category          13248 non-null  object        
 8   item_price        13248 non-null  float64       
 9   item_tracking_id  13248 non-null  int64         
 10  option_name       13248 non-null  object        
 11  option_value      13248 non-null  object        
 12  option_price      13248 non-null  float64       
 13  size              13248 non-null  object        
 14  unit_price

In [655]:
df_colddrinks.head()

,order_id,customer_id,status,cart_surcharge,order_price,item,quantity,category,item_price,item_tracking_id,option_name,option_value,option_price,size,unit_price,order_time
0,13470,776,2,0.0,12.8,Thickshake (Syrup),1,Cold Drinks,8.0,2,Flavour,Strawberry,0.0,LRG,8.0,2021-06-20 12:11:09
1,33377,9,2,0.0,12.8,Smoothie,1,Cold Drinks,7.6,2,Flavour,Green Machine,0.0,REG,7.6,2023-07-04 10:30:48
2,32604,9,2,0.0,12.8,Smoothie,1,Cold Drinks,7.6,2,Flavour,Green Machine,0.0,REG,7.6,2023-06-09 08:22:21
3,32496,9,2,0.0,12.8,Smoothie,1,Cold Drinks,7.6,2,Flavour,Green Machine,0.0,REG,7.6,2023-06-05 11:31:15
4,32399,9,2,0.0,12.8,Smoothie,1,Cold Drinks,7.6,2,Flavour,Green Machine,0.0,REG,7.6,2023-06-02 11:08:56


In [656]:
# Check if each 'option_name' for each item in an order 
# has only one corresponding 'option_value'
df_colddrinks_ = df_colddrinks[
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
].groupby(
    ['order_id', 'item_tracking_id', 'option_name']
).count().sort_values(
    'option_value', ascending=False
).reset_index(drop=False).rename(
    columns={'option_value': 'option_value_count'}
)

# Check which option(s) has more than one corresponding 'option_value'
df_colddrinks_[
    df_colddrinks_['option_value_count'] > 1
]['option_name'].unique()

array(['Ingredients'], dtype=object)

The `Ingredients` option has more than one corresponding `option_value`. Let's dive in to learn this option...

In [657]:
df_colddrinks[df_colddrinks['order_id'] == 11329][
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
]

,order_id,item_tracking_id,option_name,option_value
2776,11329,1,Ingredients,Orange
2777,11329,1,Ingredients,Lemon
2778,11329,1,Ingredients,Apple
2779,11329,1,Ingredients,Celery
2780,11329,1,Ingredients,Carrot
2781,11329,1,Ingredients,Ginger
2782,11329,1,Ingredients,Beetroot
2783,11329,2,Ingredients,Orange
2784,11329,2,Ingredients,Lemon
2785,11329,2,Ingredients,Apple


It is clear to see that some items in each order can have more than one `Ingredient`. We can concatenate these option values into a single string.

#### Concatenate `Ingredient` values for each item into single row

In [658]:
# Create a new column 'option_value_new' that concatenates ingredients
# together into a single row
df_colddrinks['option_value_new'] = df_colddrinks[
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
].groupby(
    ['order_id', 'item_tracking_id', 'option_name']
).transform(lambda x: ', '.join(x))

df_colddrinks = df_colddrinks.drop_duplicates(
    ['order_id', 'item_tracking_id', 'option_name', 'option_value_new']
)

In [659]:
# Check if the ingredients have successfully been concatenated
df_colddrinks[df_colddrinks['order_id'].isin([11329, 13470, 33377])][
    ['order_id', 'item_tracking_id', 'option_name', 
     'option_value', 'option_value_new']
]

,order_id,item_tracking_id,option_name,option_value,option_value_new
0,13470,2,Flavour,Strawberry,Strawberry
1,33377,2,Flavour,Green Machine,Green Machine
2776,11329,1,Ingredients,Orange,"Orange, Lemon, Apple, Celery, Carrot, Ginger, Beetroot"
2783,11329,2,Ingredients,Orange,"Orange, Lemon, Apple, Celery, Carrot, Ginger, Beetroot"


In [660]:
# Drop the useless 'option_value' column
df_colddrinks.drop('option_value', axis=1, inplace=True)

#### Pivot `option_name` into headers

In [661]:
# Pivot the 'option_name' column into headers, 
# using the 'option_value_new' column for their corresponding values
df_colddrinks_pivoted = df_colddrinks.pivot(
    index=['order_id', 'item_tracking_id'],
    columns='option_name',
    values='option_value_new',
).reset_index(drop=False)

In [662]:
df_colddrinks_pivoted.head()

option_name,order_id,item_tracking_id,Cream,Decaf,Equal Sugar,Extra shot,Flavour,Ice,Ingredients,Milk,Raw Sugar,Strength,Syrup,White Sugar
0,3287,1,NaN,NaN,0,0,NaN,NaN,NaN,NaN,0,Full,NaN,0
1,3295,1,NaN,NaN,0,1,NaN,NaN,NaN,NaN,0,Full,NaN,0
2,3316,3,NaN,NaN,NaN,NaN,NaN,No Ice,"Orange, Lemon, Apple, Celery, Carrot, Ginger, Beetroot",NaN,NaN,NaN,NaN,NaN
3,3316,4,NaN,NaN,NaN,NaN,Butterscotch,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3317,1,NaN,NaN,0,0,NaN,NaN,NaN,NaN,0,Full,NaN,0


In [663]:
# Gather other attributes for each item per order
df_colddrinks_attr = df_colddrinks.drop(
    ['option_name', 'option_value_new', 'option_price'], 
    axis=1,
).drop_duplicates()

# Join the two tables to create a final table for 
# cold drinks, ensuring each row represents one item per order
df_colddrinks_final = df_colddrinks_attr.merge(
    df_colddrinks_pivoted, 
    how='inner', 
    on=['order_id', 'item_tracking_id'],
)

#### Create a new `option_price` column

In [664]:
# Create a new 'option_price' column that represents 
# the difference between 'item_price' per unit and 'unit_price'
df_colddrinks_final['option_price'] = (
    df_colddrinks_final['item_price'] / df_colddrinks_final['quantity'] 
    - df_colddrinks_final['unit_price']
)

In [665]:
# Find the order that includes the highest number of cold drinks
df_colddrinks_final['order_id'].value_counts()[:1]

15001    6
Name: order_id, dtype: Int64

In [666]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 15001
df_colddrinks_final[df_colddrinks_final['order_id'] == 15001][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
]

,order_id,item_tracking_id,item,size,quantity,order_price,item_price,unit_price,option_price
5243,15001,2,Milkshake (Syrup),SML,1,25.5,3.9,3.9,0.0
5244,15001,3,Milkshake (Syrup),SML,1,25.5,3.9,3.9,0.0
5245,15001,1,Iced Mocha with Cream,REG,1,25.5,6.0,6.0,0.0
5246,15001,4,Milkshake (Syrup),SML,1,25.5,3.9,3.9,0.0
5247,15001,5,Milkshake (Syrup),SML,1,25.5,3.9,3.9,0.0
5248,15001,6,Milkshake (Syrup),SML,1,25.5,3.9,3.9,0.0


Now, let's clean all the option columns.

#### Impute missing options

In [667]:
option_columns = list(df_colddrinks['option_name'].unique())
option_columns

['Flavour',
 'Milk',
 'Strength',
 'Decaf',
 'Syrup',
 'Extra shot',
 'Cream',
 'Ingredients',
 'White Sugar',
 'Raw Sugar',
 'Equal Sugar',
 'Ice']

In [668]:
show_unique_options(df_colddrinks_final, option_columns)

,Options,Unique Values
0,Flavour,"[Strawberry, Green Machine, Mango Magic, nan, Spearmint, Cookies and Cream, Carrot, Chocolate, Banana, Orange, Blue Heaven, Lime, Caramel, Vanilla, Banana Buzz, Koko Berries, Apple, Butterscotch]"
1,Milk,"[nan, Skim, Full Cream, Oat Milk, Bonsoy, Lactose Free, Almond, Soy, Coconut Milk, Happy Soy Boy, Almond Milk Lab]"
2,Strength,"[nan, Full, Quarter, Half]"
3,Decaf,"[nan, Normal, Decaf]"
4,Syrup,"[nan, None, Vanilla]"
5,Extra shot,"[nan, 1, 0, 2, 3]"
6,Cream,"[nan, With Cream, No Cream]"
7,Ingredients,"[nan, Orange, Lemon, Celery, Beetroot, Lemon, Ginger, Orange, Lemon, Apple, Celery, Carrot, Ginger, Orange, Lemon, Apple, Celery, Carrot, Ginger, Beetroot, Orange, Lemon, Apple, Celery, Ginger, Beetroot, Orange, Apple, Celery, Carrot, Ginger, Lemon, Apple, Celery, Ginger, Beetroot, Orange, Apple, Carrot, Ginger, Orange, Lemon, Apple, Orange, Apple, Lemon, Celery, Ginger, Beetroot, Apple, Orange, Lemon, Apple, Carrot, Orange, Lemon, Apple, Celery, Lemon, Apple, Celery, Ginger, Orange, Lemon, Ginger, Orange, Lemon, Apple, Carrot, Ginger, Lemon, Orange, Lemon, Orange, Apple, Ginger, Orange, Apple, Carrot, Orange, Apple, Celery, Carrot, Lemon, Celery, Orange, Apple, Beetroot, Orange, Lemon, Apple, Celery, Ginger, Orange, Lemon, Apple, Celery, Carrot, Orange, Carrot, Ginger, Celery, Ginger, Apple, Carrot, Ginger, Orange, Celery, Carrot, Orange, Apple, Celery, Celery, Lemon, Celery, Ginger, Lemon, Apple, Carrot, Ginger, Beetroot, Orange, Lemon, Carrot, Ginger, Orange, Carrot, Orange, Celery, Ginger, Ginger, Orange, Celery, Orange, Lemon, Apple, Celery, Carrot, Beetroot, Apple, Ginger, Orange, Lemon, Celery, Ginger, Beetroot, Orange, Ginger, Apple, Celery, Orange, Lemon, Celery, Carrot, Ginger, Beetroot, Celery, Ginger, Beetroot, Lemon, Ginger, Beetroot, Orange, Lemon, Celery, Carrot, Orange, Apple, Celery, Carrot, Ginger, Beetroot, Carrot, Ginger, Beetroot, Lemon, Apple, Carrot, Ginger, Lemon, Apple, Celery, Carrot, Ginger, Orange, Apple, Celery, Ginger]"
8,White Sugar,"[nan, 0]"
9,Raw Sugar,"[nan, 0]"


In [669]:
# Check the unique options associated with each cold drink item
colddrinks_items = list(df_colddrinks['item'].unique())

colddrink_item_option_dict = {}
for item in colddrinks_items:
    colddrink_item_option_dict[item] = (
        df_colddrinks[
            df_colddrinks['item'] == item
        ]['option_name'].unique()
    )
    
print(colddrink_item_option_dict)

{'Thickshake (Syrup)': array(['Flavour'], dtype=object), 'Smoothie': array(['Flavour'], dtype=object), 'Ice Latte': array(['Milk', 'Strength', 'Decaf', 'Syrup', 'Extra shot'], dtype=object), 'Iced Coffee with Cream': array(['Milk', 'Strength', 'Decaf', 'Syrup', 'Cream', 'Extra shot'],
      dtype=object), 'Iced Mocha with Cream': array(['Milk', 'Strength', 'Decaf', 'Cream', 'Extra shot'], dtype=object), 'Ice Chai': array(['Milk', 'Strength', 'Extra shot'], dtype=object), 'Iced Chocolate with Cream': array(['Milk', 'Strength', 'Cream', 'Extra shot'], dtype=object), 'Coffee Frappe': array(['Milk', 'Strength', 'Decaf', 'Extra shot'], dtype=object), 'Milkshake (Syrup)': array(['Flavour', 'Milk'], dtype=object), 'Freshly Squeezed Juice': array(['Ingredients', 'Flavour', 'Ice'], dtype=object), 'Coffee Milkshake': array(['Milk', 'Strength', 'Decaf', 'Extra shot'], dtype=object), 'Affogato': array(['Strength', 'White Sugar', 'Raw Sugar', 'Equal Sugar',
       'Extra shot', 'Decaf'], dtype=obje

In [670]:
def show_options_per_item(df, item_option_dict, item_cat):
    # Check if any applicable option column for each item has NULl values
    items = item_option_dict.keys()
    has_null_list = []
    for item in items:
        has_null = np.any(
            pd.isna(df[item_option_dict[item]][
                df['item'] == item
            ]).to_numpy() == True
        )
        has_null_list.append(has_null)
   
    return pd.DataFrame(
        {
            item_cat: item_option_dict.keys(), 
            'Applicable Options': item_option_dict.values(),
            'Has NULL': has_null_list,
        }
    )

In [671]:
show_options_per_item(
    df_colddrinks_final, colddrink_item_option_dict, 'Cold Drinks'
)

,Cold Drinks,Applicable Options,Has NULL
0,Thickshake (Syrup),[Flavour],False
1,Smoothie,[Flavour],False
2,Ice Latte,"[Milk, Strength, Decaf, Syrup, Extra shot]",True
3,Iced Coffee with Cream,"[Milk, Strength, Decaf, Syrup, Cream, Extra shot]",True
4,Iced Mocha with Cream,"[Milk, Strength, Decaf, Cream, Extra shot]",True
5,Ice Chai,"[Milk, Strength, Extra shot]",False
6,Iced Chocolate with Cream,"[Milk, Strength, Cream, Extra shot]",False
7,Coffee Frappe,"[Milk, Strength, Decaf, Extra shot]",False
8,Milkshake (Syrup),"[Flavour, Milk]",True
9,Freshly Squeezed Juice,"[Ingredients, Flavour, Ice]",True


In [672]:
# Impute the applicable option columns only for each item with the most common choice
has_null_list = []
for item in colddrinks_items:
    # Find the list of applicable option columns for the current item
    item_options = colddrink_item_option_dict[item]
    for option in option_columns:
        # If the current option column is applicable for the current item
        if option in item_options:
            # Get the most frequent choice for the option
            most_common = df_colddrinks_final[
                df_colddrinks_final['item'] == item
            ][option].value_counts().index[0]
            
            # Impute the NaN in the option column with the most frequent choice
            df_colddrinks_final.loc[
                df_colddrinks_final[
                    (df_colddrinks_final['item'] == item)
                    & (pd.isna(df_colddrinks_final[option]))
                ].index, option
            ] = most_common

In [673]:
show_options_per_item(
    df_colddrinks_final, colddrink_item_option_dict, 'Cold Drinks'
)

,Cold Drinks,Applicable Options,Has NULL
0,Thickshake (Syrup),[Flavour],False
1,Smoothie,[Flavour],False
2,Ice Latte,"[Milk, Strength, Decaf, Syrup, Extra shot]",False
3,Iced Coffee with Cream,"[Milk, Strength, Decaf, Syrup, Cream, Extra shot]",False
4,Iced Mocha with Cream,"[Milk, Strength, Decaf, Cream, Extra shot]",False
5,Ice Chai,"[Milk, Strength, Extra shot]",False
6,Iced Chocolate with Cream,"[Milk, Strength, Cream, Extra shot]",False
7,Coffee Frappe,"[Milk, Strength, Decaf, Extra shot]",False
8,Milkshake (Syrup),"[Flavour, Milk]",False
9,Freshly Squeezed Juice,"[Ingredients, Flavour, Ice]",False


In [674]:
show_unique_options(df_colddrinks_final, option_columns)

,Options,Unique Values
0,Flavour,"[Strawberry, Green Machine, Mango Magic, nan, Spearmint, Cookies and Cream, Orange, Carrot, Chocolate, Banana, Blue Heaven, Lime, Caramel, Vanilla, Banana Buzz, Koko Berries, Apple, Butterscotch]"
1,Milk,"[nan, Skim, Full Cream, Oat Milk, Bonsoy, Lactose Free, Almond, Soy, Coconut Milk, Happy Soy Boy, Almond Milk Lab]"
2,Strength,"[nan, Full, Quarter, Half]"
3,Decaf,"[nan, Normal, Decaf]"
4,Syrup,"[nan, None, Vanilla]"
5,Extra shot,"[nan, 1, 0, 2, 3]"
6,Cream,"[nan, With Cream, No Cream]"
7,Ingredients,"[nan, Orange, Lemon, Celery, Beetroot, Lemon, Ginger, Orange, Lemon, Apple, Celery, Carrot, Ginger, Orange, Lemon, Apple, Celery, Carrot, Ginger, Beetroot, Orange, Lemon, Apple, Celery, Ginger, Beetroot, Orange, Apple, Celery, Carrot, Ginger, Lemon, Apple, Celery, Ginger, Beetroot, Orange, Apple, Carrot, Ginger, Orange, Lemon, Apple, Orange, Apple, Lemon, Celery, Ginger, Beetroot, Apple, Orange, Lemon, Apple, Carrot, Orange, Lemon, Apple, Celery, Lemon, Apple, Celery, Ginger, Orange, Lemon, Ginger, Orange, Lemon, Apple, Carrot, Ginger, Lemon, Orange, Lemon, Orange, Apple, Ginger, Orange, Apple, Carrot, Orange, Apple, Celery, Carrot, Lemon, Celery, Orange, Apple, Beetroot, Orange, Lemon, Apple, Celery, Ginger, Orange, Lemon, Apple, Celery, Carrot, Orange, Carrot, Ginger, Celery, Ginger, Apple, Carrot, Ginger, Orange, Celery, Carrot, Orange, Apple, Celery, Celery, Lemon, Celery, Ginger, Lemon, Apple, Carrot, Ginger, Beetroot, Orange, Lemon, Carrot, Ginger, Orange, Carrot, Orange, Celery, Ginger, Ginger, Orange, Celery, Orange, Lemon, Apple, Celery, Carrot, Beetroot, Apple, Ginger, Orange, Lemon, Celery, Ginger, Beetroot, Orange, Ginger, Apple, Celery, Orange, Lemon, Celery, Carrot, Ginger, Beetroot, Celery, Ginger, Beetroot, Lemon, Ginger, Beetroot, Orange, Lemon, Celery, Carrot, Orange, Apple, Celery, Carrot, Ginger, Beetroot, Carrot, Ginger, Beetroot, Lemon, Apple, Carrot, Ginger, Lemon, Apple, Celery, Carrot, Ginger, Orange, Apple, Celery, Ginger]"
8,White Sugar,"[nan, 0]"
9,Raw Sugar,"[nan, 0]"


Since the `Whige Sugar`, `Raw Sugar`, `Equal Sugar` columns only contains either 0 or NaN values, we can safely remove those option columns to streamline the table.

In [675]:
df_colddrinks_final.drop(
    ['White Sugar', 'Raw Sugar', 'Equal Sugar'],
    axis=1,
    inplace=True,
)

In [676]:
print(df_colddrinks_final.shape)
print(df_colddrinks_final.columns)

(5439, 23)
Index(['order_id', 'customer_id', 'status', 'cart_surcharge', 'order_price',
       'item', 'quantity', 'category', 'item_price', 'item_tracking_id',
       'size', 'unit_price', 'order_time', 'Cream', 'Decaf', 'Extra shot',
       'Flavour', 'Ice', 'Ingredients', 'Milk', 'Strength', 'Syrup',
       'option_price'],
      dtype='object')


#### Export the Cold Drinks data as a separate CSV file

In [677]:
df_colddrinks_final.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_colddrinks_pivoted_mx.csv',
    header=True,
    index=False,
)

### Transform Kitchen Data for Better Usability

In [678]:
df_kitchen.reset_index(drop=True, inplace=True)
print(df_kitchen.shape)
print(df_kitchen.info())

(14625, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14625 entries, 0 to 14624
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          14625 non-null  Int64         
 1   customer_id       14625 non-null  Int64         
 2   status            14625 non-null  Int64         
 3   cart_surcharge    14625 non-null  float64       
 4   order_price       14625 non-null  float64       
 5   item              14625 non-null  object        
 6   quantity          14625 non-null  int64         
 7   category          14625 non-null  object        
 8   item_price        14625 non-null  float64       
 9   item_tracking_id  14625 non-null  int64         
 10  option_name       14625 non-null  object        
 11  option_value      14625 non-null  object        
 12  option_price      14625 non-null  float64       
 13  size              14625 non-null  object        
 14  unit_price

In [679]:
# Check the unique options for Kitchen items
df_kitchen['option_name'].unique()

array(['Bread', 'Extras', 'Heated', 'Toppings', 'Options', 'Pasta',
       'Extra'], dtype=object)

The `Extras` and `Extra` options seem to be the same. Let's investigate further to identify which items have the `Extra` / `Extras` option.

In [680]:
# Identify items that have the 'Extra' option
df_kitchen[df_kitchen['option_name'] == 'Extra']['item'].unique()

array(['Acai Smoothie bowl (VO)'], dtype=object)

In [681]:
# Identify items that have the 'Extras' option
df_kitchen[df_kitchen['option_name'] == 'Extras']['item'].unique()

array(['Toastie', 'Toasted Roll', 'Croissant', 'Build Your Own Breakfast',
       'Hot Potatoes', 'Big Breakfast', 'Plain Toast'], dtype=object)

Since only one item has the `Extra` option, we can rename it to `Extras` to avoid confusion.

In [682]:
# Rename 'Extra' option to 'Extras'
df_kitchen['option_name'] = df_kitchen['option_name'].apply(
    lambda x: 'Extras' if x == 'Extra' else x
)

In [683]:
# Re-check the unique options for Kitchen items
df_kitchen['option_name'].unique()

array(['Bread', 'Extras', 'Heated', 'Toppings', 'Options', 'Pasta'],
      dtype=object)

In [684]:
# Check if each 'option_name' for each item in an order 
# has only one corresponding 'option_value'
df_kitchen_ = df_kitchen[
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
].groupby(
    ['order_id', 'item_tracking_id', 'option_name']
).count().sort_values(
    'option_value', ascending=False
).reset_index(drop=False).rename(
    columns={'option_value': 'option_value_count'}
)

df_kitchen_[
    df_kitchen_['option_value_count'] > 1
]['option_name'].unique()

array(['Extras', 'Toppings', 'Options'], dtype=object)

The `Extras`, `Toppings`, and `Options` option have more than one corresponding `option_value`. Let's dive in to learn this option...

In [685]:
# Sample an item in an order that contains more than one 'Extras'
df_kitchen_[
    (
        df_kitchen_['option_name'] == 'Extras'
    )
    & (
        df_kitchen_['option_value_count'] > 1
    ) 
].head(1)

,order_id,item_tracking_id,option_name,option_value_count
0,4378,6,Extras,14


In [686]:
df_kitchen[
    (df_kitchen['order_id'] == 4378)
    & (df_kitchen['item_tracking_id'] == 6)
][
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
]

,order_id,item_tracking_id,option_name,option_value
11072,4378,6,Bread,Gluten Free
11073,4378,6,Extras,Extra Egg
11074,4378,6,Extras,Bacon
11075,4378,6,Extras,Grilled Tomato
11076,4378,6,Extras,Hollandaise
11077,4378,6,Extras,Hash Brown
11078,4378,6,Extras,Spanish Chorizo
11079,4378,6,Extras,Smashed Avo and Feta
11080,4378,6,Extras,Sauteed Mushrooms
11081,4378,6,Extras,Avocado


In [687]:
# Sample an item in an order that contains more than one 'Toppings'
df_kitchen_[
    (
        df_kitchen_['option_name'] == 'Toppings'
    )
    & (
        df_kitchen_['option_value_count'] > 1
    ) 
].head(1)

,order_id,item_tracking_id,option_name,option_value_count
1767,15178,2,Toppings,2


In [688]:
df_kitchen[
    (df_kitchen['order_id'] == 15178)
    & (df_kitchen['item_tracking_id'] == 2)
][
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
]

,order_id,item_tracking_id,option_name,option_value
3454,15178,2,Toppings,Banana
3455,15178,2,Toppings,Strawberries


In [689]:
# Sample an item in an order that contains more than one 'Options'
df_kitchen_[
    (
        df_kitchen_['option_name'] == 'Options'
    )
    & (
        df_kitchen_['option_value_count'] > 1
    ) 
].head(1)

,order_id,item_tracking_id,option_name,option_value_count
1941,4378,17,Options,2


In [690]:
df_kitchen[
    (df_kitchen['order_id'] == 4378)
    & (df_kitchen['item_tracking_id'] == 17)
][
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
]

,order_id,item_tracking_id,option_name,option_value
11069,4378,17,Options,No Mable Syrup
11070,4378,17,Options,Hashbrown


#### Concatenate `Extras`, `Toppings` and `Options` values for each item into single row

In [691]:
# Create a new column 'option_value_new' that concatenates ingredients
# together into a single row
df_kitchen['option_value_new'] = df_kitchen[
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
].groupby(
    ['order_id', 'item_tracking_id', 'option_name']
).transform(lambda x: ', '.join(x))

df_kitchen = df_kitchen.drop_duplicates(
    ['order_id', 'item_tracking_id', 'option_name', 'option_value_new']
)

In [692]:
# Check if the ingredients have successfully been concatenated
df_kitchen[df_kitchen['order_id'].isin([4378, 15178, 33377])][
    ['order_id', 'item_tracking_id', 'option_name', 
     'option_value', 'option_value_new']
].sort_values(['order_id', 'item_tracking_id'], ascending=[True, True])

,order_id,item_tracking_id,option_name,option_value,option_value_new
11104,4378,1,Pasta,Spaghetti,Spaghetti
11105,4378,2,Pasta,Fettuccini,Fettuccini
11102,4378,3,Pasta,Spaghetti,Spaghetti
11103,4378,4,Pasta,Gnocchi,Gnocchi
11072,4378,6,Bread,Gluten Free,Gluten Free
11073,4378,6,Extras,Extra Egg,"Extra Egg, Bacon, Grilled Tomato, Hollandaise, Hash Brown, Spanish Chorizo, Smashed Avo and Feta, Sauteed Mushrooms, Avocado, Baby Spinach, Halloumi, Baked Beans, Smoked Salmon, Pulled Pork"
11087,4378,8,Bread,Gluten Free,Gluten Free
11088,4378,8,Extras,Extra Egg,"Extra Egg, Bacon, Grilled Tomato, Hollandaise, Hash Brown, Spanish Chorizo, Smashed Avo and Feta, Sauteed Mushrooms, Avocado, Baby Spinach, Halloumi, Baked Beans, Smoked Salmon, Pulled Pork"
11069,4378,17,Options,No Mable Syrup,"No Mable Syrup, Hashbrown"
11071,4378,27,Bread,Gluten Free,Gluten Free


In [693]:
# Drop the useless 'option_value' column
df_kitchen.drop('option_value', axis=1, inplace=True)

#### Pivot `option_name` into headers

In [694]:
# Pivot the 'option_name' column into headers, 
# using the 'option_value_new' column for their corresponding values
df_kitchen_pivoted = df_kitchen.pivot(
    index=['order_id', 'item_tracking_id'],
    columns='option_name',
    values='option_value_new',
).reset_index(drop=False)

In [695]:
df_kitchen_pivoted.head()

option_name,order_id,item_tracking_id,Bread,Extras,Heated,Options,Pasta,Toppings
0,3313,1,White,NaN,NaN,NaN,NaN,NaN
1,3313,2,Sourdough,NaN,NaN,NaN,NaN,NaN
2,3315,2,White,NaN,NaN,NaN,NaN,NaN
3,3316,6,Sourdough,NaN,NaN,NaN,NaN,NaN
4,3317,9,White,NaN,NaN,NaN,NaN,NaN


In [696]:
# Gather other attributes for each item per order
df_kitchen_attr = df_kitchen.drop(
    ['option_name', 'option_value_new', 'option_price'], 
    axis=1,
).drop_duplicates()

# Join the two tables to create a final table for kitchen,
# ensuring each row represents one item per order
df_kitchen_final = df_kitchen_attr.merge(
    df_kitchen_pivoted, 
    how='inner', 
    on=['order_id', 'item_tracking_id'],
)

#### Create a new `option_price` column

In [697]:
# Create a new 'option_price' column that represents 
# the difference between 'item_price' per unit and 'unit_price'
df_kitchen_final['option_price'] = (
    df_kitchen_final['item_price'] / df_kitchen_final['quantity'] 
    - df_kitchen_final['unit_price']
)

In [698]:
# Find the order that includes the highest number of kitchen items
df_kitchen_final['order_id'].value_counts()[:1]

28760    9
Name: order_id, dtype: Int64

In [699]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 28760
df_kitchen_final[df_kitchen_final['order_id'] == 28760][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
]

,order_id,item_tracking_id,item,size,quantity,order_price,item_price,unit_price,option_price
7559,28760,1,Toasted Roll,BACON-EGG-CHEESE,1,99.5,11.5,9.5,2.0
7560,28760,2,Toasted Roll,BACON-EGG-CHEESE,1,99.5,11.5,9.5,2.0
7561,28760,3,Toasted Roll,BACON-EGG-CHEESE,1,99.5,11.5,9.5,2.0
7562,28760,4,Toasted Roll,BACON-EGG-CHEESE,1,99.5,11.5,9.5,2.0
7563,28760,5,Toasted Roll,BACON-EGG-CHEESE,1,99.5,11.5,9.5,2.0
7564,28760,6,Toasted Roll,BACON-EGG-CHEESE,1,99.5,11.5,9.5,2.0
7565,28760,7,Toasted Roll,BACON-EGG-CHEESE,1,99.5,11.5,9.5,2.0
7566,28760,8,Toasted Roll,BACON-EGG-CHEESE,1,99.5,11.5,9.5,2.0
7567,28760,9,Toastie,EGG,1,99.5,7.5,7.5,0.0


Now, let's clean all the option columns.

#### Impute missing options

In [700]:
option_columns = list(df_kitchen['option_name'].unique())
option_columns

['Bread', 'Extras', 'Heated', 'Toppings', 'Options', 'Pasta']

In [701]:
show_unique_options(df_kitchen_final, option_columns)

,Options,Unique Values
0,Bread,"[Multigrain, White, nan, Sourdough, Dark Rye, Gluten Free]"
1,Extras,"[nan, Tomato Sauce, BBQ Sauce, Salt and Pepper, Avocado, BBQ Sauce, Salt and Pepper, Tomato Sauce, Avocado, BBQ Sauce, Salt and Pepper, Tomato Sauce, Hash Brown, Salt and Pepper, Tomato Sauce, Hash Brown, Salt and Pepper, Avocado, Salt and Pepper, Tomato Sauce, Hash Brown, BBQ Sauce, Hash Brown, Bacon, Hollandaise, Sauteed Mushrooms, Tomato Sauce, Salt and Pepper, BBQ Sauce, Hash Brown, Salt and Pepper, BBQ Sauce, Avocado, peanut butter, Tomato Sauce, BBQ Sauce, Hash Brown, Hash Brown, Smashed Avo and Feta, Tomato Sauce, Avocado, Hash Brown, Bacon, Bacon, Hollandaise, Hash Brown, Smashed Avo and Feta, Halloumi, Bacon, Grilled Tomato, Avocado, Hollandaise, Hash Brown, Smashed Avo and Feta, Tomato Sauce, Avocado, Salt and Pepper, Avocado, Hash Brown, Salt and Pepper, Bacon, Avocado, Smashed Avo and Feta, Baby Spinach, Sauteed Mushrooms, Avocado, Hash Brown, Bacon, Hash Brown, Avocado, Tomato Sauce, BBQ Sauce, Avocado, Hash Brown, Sausage, Bacon, Grilled Tomato, Hash Brown, Avocado, Halloumi, Bacon, Grilled Tomato, Smashed Avo and Feta, Sauteed Mushrooms, Pulled Pork, Halloumi, Baked Beans, Extra Egg, Bacon, Avocado, Sausage, Spanish Chorizo, Avocado, Halloumi, BBQ Sauce, Avocado, Hash Brown, Tomato Sauce, BBQ Sauce, Hash Brown, Salt and Pepper, Hollandaise, Avocado, Bacon, Baked Beans, Bacon, Hash Brown, Smashed Avo and Feta, Extra Egg, Bacon, Halloumi, Bacon, Avocado, Halloumi, Tomato Sauce, Avocado, Hash Brown, Salt and Pepper, Hollandaise, Hash Brown, Avocado, Baby Spinach, Bacon, Hollandaise, Hash Brown, Avocado, Extra Egg, Bacon, Hollandaise, Hash Brown, Smashed Avo and Feta, Pulled Pork, Hollandaise, Bacon, Grilled Tomato, Smashed Avo and Feta, Bacon, Hash Brown, Extra Egg, Grilled Tomato, Hollandaise, Hash Brown, Smashed Avo and Feta, Sauteed Mushrooms, Baby Spinach, Baked Beans, Bacon, Spanish Chorizo, Sauteed Mushrooms, Avocado, Hash Brown, Avocado, Avocado, Baked Beans, Bacon, Smashed Avo and Feta, Sauteed Mushrooms, Bacon, Hollandaise, Avocado, Baby Spi..."
2,Heated,"[nan, No, Yes]"
3,Toppings,"[nan, Strawberries, Banana, Banana, Strawberries]"
4,Options,"[nan, Hashbrown, No Mable Syrup, Sweet and sour sauce, No Mable Syrup, Hashbrown]"
5,Pasta,"[nan, Spaghetti, Fettuccini, Gnocchi]"


In [702]:
# Check the unique options associated with each kitchen item
kitchen_items = list(df_kitchen['item'].unique())

kitchen_item_option_dict = {}
for item in kitchen_items:
    kitchen_item_option_dict[item] = (
        df_kitchen[
            df_kitchen['item'] == item
        ]['option_name'].unique()
    )
    
print(kitchen_item_option_dict)

{'Toastie': array(['Bread', 'Extras'], dtype=object), 'Toasted Roll': array(['Extras'], dtype=object), 'Muffin': array(['Heated'], dtype=object), 'Kids Pancakes': array(['Toppings'], dtype=object), 'Croissant': array(['Extras'], dtype=object), 'French Toast with Streaky Bacon': array(['Options'], dtype=object), 'Build Your Own Breakfast': array(['Bread', 'Extras'], dtype=object), 'Eggs Benedict': array(['Options'], dtype=object), '(Main) Pasta': array(['Pasta'], dtype=object), 'Kids Crepes': array(['Toppings'], dtype=object), 'Acai Smoothie bowl (VO)': array(['Extras'], dtype=object), 'Eggs and Bacon': array(['Bread'], dtype=object), 'Big Breakfast': array(['Bread', 'Extras'], dtype=object), 'Hot Potatoes': array(['Extras'], dtype=object), '(Entree) Pasta': array(['Pasta'], dtype=object), 'Eggs Benny': array(['Options'], dtype=object), 'Pork Benny': array(['Options'], dtype=object), 'Eggs Florentine': array(['Options'], dtype=object), 'Plain Toast': array(['Extras'], dtype=object), 'De

In [703]:
show_options_per_item(
    df_kitchen_final, kitchen_item_option_dict, 'Kitchen Items'
)

,Kitchen Items,Applicable Options,Has NULL
0,Toastie,"[Bread, Extras]",True
1,Toasted Roll,[Extras],False
2,Muffin,[Heated],False
3,Kids Pancakes,[Toppings],False
4,Croissant,[Extras],False
5,French Toast with Streaky Bacon,[Options],False
6,Build Your Own Breakfast,"[Bread, Extras]",True
7,Eggs Benedict,[Options],False
8,(Main) Pasta,[Pasta],False
9,Kids Crepes,[Toppings],False


In [704]:
# Impute the applicable option columns only for each item with the most common choice
has_null_list = []
for item in kitchen_items:
    # Find the list of applicable option columns for the current item
    item_options = kitchen_item_option_dict[item]
    for option in option_columns:
        # If the current option column is applicable for the current item
        if option in item_options:
            # Get the most frequent choice for the option
            most_common = df_kitchen_final[
                df_kitchen_final['item'] == item
            ][option].value_counts().index[0]
            
            # Impute the NaN in the option column with the most frequent choice
            df_kitchen_final.loc[
                df_kitchen_final[
                    (df_kitchen_final['item'] == item)
                    & (pd.isna(df_kitchen_final[option]))
                ].index, option
            ] = most_common

In [705]:
show_options_per_item(
    df_kitchen_final, kitchen_item_option_dict, 'Kitchen Items'
)

,Kitchen Items,Applicable Options,Has NULL
0,Toastie,"[Bread, Extras]",False
1,Toasted Roll,[Extras],False
2,Muffin,[Heated],False
3,Kids Pancakes,[Toppings],False
4,Croissant,[Extras],False
5,French Toast with Streaky Bacon,[Options],False
6,Build Your Own Breakfast,"[Bread, Extras]",False
7,Eggs Benedict,[Options],False
8,(Main) Pasta,[Pasta],False
9,Kids Crepes,[Toppings],False


In [706]:
show_unique_options(df_kitchen_final, option_columns)

,Options,Unique Values
0,Bread,"[Multigrain, White, nan, Sourdough, Dark Rye, Gluten Free]"
1,Extras,"[Tomato Sauce, Tomato Sauce, BBQ Sauce, Salt and Pepper, Avocado, BBQ Sauce, Salt and Pepper, Tomato Sauce, Avocado, nan, BBQ Sauce, Salt and Pepper, Hash Brown, Salt and Pepper, Tomato Sauce, Hash Brown, Salt and Pepper, Avocado, Salt and Pepper, Tomato Sauce, Hash Brown, BBQ Sauce, Hash Brown, Bacon, Hollandaise, Sauteed Mushrooms, Tomato Sauce, Salt and Pepper, BBQ Sauce, Hash Brown, Salt and Pepper, BBQ Sauce, Avocado, peanut butter, Tomato Sauce, BBQ Sauce, Hollandaise, Hash Brown, Hash Brown, Smashed Avo and Feta, Tomato Sauce, Avocado, Hash Brown, Bacon, Bacon, Hollandaise, Hash Brown, Smashed Avo and Feta, Halloumi, Bacon, Grilled Tomato, Avocado, Hollandaise, Hash Brown, Smashed Avo and Feta, Tomato Sauce, Avocado, Salt and Pepper, Avocado, Hash Brown, Salt and Pepper, Bacon, Avocado, Smashed Avo and Feta, Baby Spinach, Sauteed Mushrooms, Avocado, Hash Brown, Bacon, Hash Brown, Avocado, Tomato Sauce, BBQ Sauce, Avocado, Hash Brown, Sausage, Bacon, Grilled Tomato, Hash Brown, Avocado, Halloumi, Bacon, Grilled Tomato, Smashed Avo and Feta, Sauteed Mushrooms, Pulled Pork, Halloumi, Baked Beans, Extra Egg, Bacon, Avocado, Sausage, Spanish Chorizo, Avocado, Halloumi, BBQ Sauce, Avocado, Hash Brown, Tomato Sauce, BBQ Sauce, Hash Brown, Salt and Pepper, Hollandaise, Avocado, Bacon, Baked Beans, Bacon, Hash Brown, Smashed Avo and Feta, Extra Egg, Bacon, Halloumi, Bacon, Avocado, Halloumi, Tomato Sauce, Avocado, Hash Brown, Salt and Pepper, Hollandaise, Hash Brown, Avocado, Baby Spinach, Bacon, Hollandaise, Hash Brown, Avocado, Extra Egg, Bacon, Hollandaise, Hash Brown, Smashed Avo and Feta, Pulled Pork, Bacon, Grilled Tomato, Smashed Avo and Feta, Bacon, Hash Brown, Extra Egg, Grilled Tomato, Hollandaise, Hash Brown, Smashed Avo and Feta, Sauteed Mushrooms, Baby Spinach, Baked Beans, Bacon, Spanish Chorizo, Sauteed Mushrooms, Avocado, Hash Brown, Avocado, Avocado, Baked Beans, Bacon, Smashed Avo and Feta, Sauteed Mushrooms, Bacon, Hollandaise, Avocado, Baby Spi..."
2,Heated,"[nan, No, Yes]"
3,Toppings,"[nan, Strawberries, Banana, Banana, Strawberries]"
4,Options,"[nan, Hashbrown, No Mable Syrup, Sweet and sour sauce, No Mable Syrup, Hashbrown]"
5,Pasta,"[nan, Spaghetti, Fettuccini, Gnocchi]"


In [707]:
print(df_kitchen_final.shape)
print(df_kitchen_final.info())

(8470, 20)
Index(['order_id', 'customer_id', 'status', 'cart_surcharge', 'order_price',
       'item', 'quantity', 'category', 'item_price', 'item_tracking_id',
       'size', 'unit_price', 'order_time', 'Bread', 'Extras', 'Heated',
       'Options', 'Pasta', 'Toppings', 'option_price'],
      dtype='object')


#### Export the Kitchen data as a separate CSV file

In [708]:
df_kitchen_final.to_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_kitchen_pivoted_mx.csv',
    header=True,
    index=False,
)

## Union all Pivoted Dataframes into a Single One

In [709]:
df_pivoted = pd.concat(
    [
        df_hotdrinks_final,
        df_colddrinks_final,
        df_kitchen_final,
    ],
    axis=0,
    ignore_index=True,
)

In [711]:
print(df_pivoted.shape)
print(df_pivoted.info())

(66233, 34)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66233 entries, 0 to 66232
Data columns (total 34 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          66233 non-null  Int64         
 1   customer_id       66233 non-null  Int64         
 2   status            66233 non-null  Int64         
 3   cart_surcharge    66233 non-null  float64       
 4   order_price       66233 non-null  float64       
 5   item              66233 non-null  object        
 6   quantity          66233 non-null  int64         
 7   category          66233 non-null  object        
 8   item_price        66233 non-null  float64       
 9   item_tracking_id  66233 non-null  int64         
 10  size              66233 non-null  object        
 11  unit_price        66233 non-null  float64       
 12  order_time        66233 non-null  datetime64[ns]
 13  Decaf             53361 non-null  object        
 14  Equal Suga